In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install chess
!pip install numba
!apt-get install stockfish

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 68.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147879 sha256=830edd637174b7d1c5a189e25e8952825fbcd59353340b63ec068875fc2c3b79
  Stored in directory: /root/.cache/pip/wheels/74/65/99/8c3a88ba852c1ed8d34daed144b8cb4c654d638677483d19ff
Successfully built chess
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  polyglot xboard | scid
The following NEW packages will be installed:
  stockfish
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 33.5 MB of archives.
After this operation, 40.4 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 stockfish amd64 16-1build1 [33.5 MB]
Fetched 33.5 MB in 3s (12.8 MB/s)
Selecting previously unselected package stockfish.
(Reading database ... 127332 files and directories

In [3]:
from numba import njit
import numpy as np
import chess, chess.polyglot
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, random
import os
import time
import collections
from tqdm import trange

@njit(cache=True, fastmath=True, nogil=True)
def _encode_state_numba(piece_map_squares: np.ndarray, piece_map_symbols: np.ndarray,
                        turn_is_white: bool, castling_possible: bool) -> np.ndarray:
    """Numba helper for get_encoded_state. Needs simplified input."""
    board_array = np.zeros((14, 8, 8), dtype=np.float32) # Channels first
    piece_map_codes = np.array([
        ord('P'), ord('N'), ord('B'), ord('R'), ord('Q'), ord('K'),
        ord('p'), ord('n'), ord('b'), ord('r'), ord('q'), ord('k')
    ], dtype=np.int32)
    piece_map_indices = np.arange(12, dtype=np.int32)

    for i in range(len(piece_map_squares)):
        square = piece_map_squares[i]
        symbol_ord = piece_map_symbols[i] # Get piece symbol ordinal

        rank = 7 - (square // 8)
        col = square % 8

        # Find the layer index corresponding to the piece symbol
        layer = -1
        for j in range(len(piece_map_codes)):
            if symbol_ord == piece_map_codes[j]:
                layer = piece_map_indices[j]
                break

        if layer != -1:
            board_array[layer, rank, col] = 1.0

    # Channel 12: Turn information
    board_array[12, :, :] = 1.0 if turn_is_white else 0.0
    # Channel 13: Castling rights
    board_array[13, :, :] = 1.0 if castling_possible else 0.0

    return board_array

@njit(cache=True, fastmath=True, nogil=True)
def _apply_dirichlet_noise(priors: np.ndarray, valid_mask: np.ndarray,
                           alpha: float, epsilon: float) -> np.ndarray:
    """Applies noise and renormalizes (extracted from MCTS.run)."""
    action_size = len(priors)
    noise = np.random.dirichlet(np.full(action_size, alpha, dtype=np.float64))
    masked = (1.0 - epsilon) * priors + epsilon * noise
    masked *= valid_mask # Apply valid mask
    sum_masked = np.sum(masked)
    if sum_masked > 1e-8:
        masked /= sum_masked
    else: # Handle case where masking zeros everything
        num_valid = np.sum(valid_mask)
        if num_valid > 0:
             masked = valid_mask / num_valid
        # else: return zero array, should not happen if valid moves exist
    return masked

@njit(cache=True, fastmath=True, nogil=True)
def _calculate_final_probs(counts: np.ndarray, valid_mask: np.ndarray, temp: float) -> np.ndarray:
    """Calculates final policy probabilities (extracted from MCTS.run)."""
    probs = np.zeros_like(counts, dtype=np.float32)
    if temp < 1e-3:
        legal_idxs = np.where(valid_mask == 1)[0]
        if len(legal_idxs) == 0: return probs # No legal moves
        best_action = legal_idxs[np.argmax(counts[legal_idxs])]
        probs[best_action] = 1.0
    else:
        # Use float64 for intermediate exponentiation for stability
        masked_counts = counts.astype(np.float64) * valid_mask
        sum_counts = np.sum(masked_counts)
        if sum_counts > 1e-8: # Check if there are valid counts > 0
            probs_temp = masked_counts**(1.0 / temp)
            sum_probs = np.sum(probs_temp)
            if sum_probs > 1e-8:
                 probs = (probs_temp / sum_probs).astype(np.float32)
            else: # Counts were valid but low, temp made them ~= 0
                 num_valid = np.sum(valid_mask)
                 if num_valid > 0: probs = (valid_mask / num_valid).astype(np.float32)
        else: # All valid counts were 0
            num_valid = np.sum(valid_mask)
            if num_valid > 0: probs = (valid_mask / num_valid).astype(np.float32)
            # else: return zero array
    return probs

@njit(cache=True, fastmath=True, nogil=True) # Use @numba.njit if you didn't use 'from numba import njit'
def select_ucb_array(Q_vals: np.ndarray, N_sa: np.ndarray, N_s: int,
                     P_prior: np.ndarray, C_puct: float, valid_moves_mask: np.ndarray) -> int:
    """
    Selects the action with the highest UCB score using NumPy arrays and Numba.
    Args:
        Q_vals: Array of Q-values for actions (size = action_size)
        N_sa: Array of visit counts for state-action pairs (size = action_size)
        N_s: Visit count for the state
        P_prior: Array of prior probabilities for actions (size = action_size)
        C_puct: Exploration constant
        valid_moves_mask: Binary array indicating valid moves (size = action_size)

    Returns:
        The index of the best action, or -1 if no valid action found.
    """
    max_ucb = -np.inf
    best_action = -1
    # Precompute sqrt(N(s)) to avoid repeated calculation inside loop
    # Add a small epsilon to N_s before sqrt to prevent division by zero if N_s is 0 initially,
    # although N_s should ideally be at least 1 when selection happens after expansion.
    # However, let's handle N_s=0 case explicitly.
    sqrt_Ns = math.sqrt(float(N_s)) if N_s > 0 else 0.0

    action_size = len(Q_vals)
    for action in range(action_size):
        if valid_moves_mask[action] == 1: # Only consider valid moves
            # U = Q(s,a) + C_puct * P(s,a) * sqrt(N(s)) / (1 + N(s,a))
            # Use Numba compatible math.sqrt or ensure NumPy is available if using np.sqrt
            ucb = Q_vals[action] + C_puct * P_prior[action] * sqrt_Ns / (1.0 + N_sa[action])

            if ucb > max_ucb:
                max_ucb = ucb
                best_action = action

    # This case should ideally not be reached if valid_moves_mask guarantees at least one valid move
    if best_action == -1:
         valid_indices = np.where(valid_moves_mask == 1)[0]
         if len(valid_indices) > 0:
              # Fallback: Maybe choose the valid move with the highest prior? Or just the first valid one?
              # Let's choose the one with the highest prior probability among valid ones.
              best_action = valid_indices[np.argmax(P_prior[valid_indices])]
              # Or simply: best_action = valid_indices[0]

    return best_action

@njit(cache=True, fastmath=True, nogil=True)
def _ucb_score(child_W: float, child_N: int, parent_N: int, child_P: float, C: float) -> float:
    """
    Calculates the UCB score for a child node using Numba.
    Args:
        child_W: Total action value (W) accumulated for the child from the parent's perspective.
        child_N: Visit count (N) for the state-action pair leading to the child.
        parent_N: Visit count (N) for the parent state.
        child_P: Prior probability (P) of the action leading to the child.
        C: Exploration constant.
    Returns:
        UCB score.
    """
    # Q value estimate: W / N
    q_value = child_W / child_N if child_N > 0 else 0.0
    # Exploration bonus
    exploration_term = C * child_P * (math.sqrt(float(parent_N)) / (1.0 + child_N))
    return q_value + exploration_term

In [4]:
CHECKPOINT_DIR = "/content/drive/MyDrive/AlphaZero_Checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/checkpoint.pt"

def save_checkpoint(model, optimizer, episode, path=CHECKPOINT_PATH):
    torch.save({
        'episode': episode,
        'model_state': model.state_dict(),
        'optim_state': optimizer.state_dict(),
    }, path)
    print(f"Checkpoint saved to {path} (episode {episode})")

def list_checkpoints(checkpoint_dir=CHECKPOINT_DIR):
    """
    Returns a sorted list of all .pt/.pth files in `checkpoint_dir` and
    prints them with an index so you can choose one.
    """
    if not os.path.isdir(checkpoint_dir):
        print(f"[!] No such directory: {checkpoint_dir}")
        return []

    files = [f for f in os.listdir(checkpoint_dir) if f.lower().endswith(".pt")]
    files.sort()
    if not files:
        print("No checkpoint files found.")
        return []

    print("Available checkpoints:")
    for i, fname in enumerate(files):
        print(f"  [{i}] {fname}")
    return files

def choose_and_load_checkpoint(model, optimizer=None, mode=False, num_residual_blocks=0, num_hidden_channels=0, checkpoint_dir=CHECKPOINT_DIR, device=None):
    """
    Lists checkpoints, prompts the user to select one, then loads it.
    Returns the episode number to resume from.
    """
    files = list_checkpoints(checkpoint_dir)
    if not files:
        print("No Drive checkpoint found, starting fresh.")
        return 0

    idx = None
    while idx is None:
        try:
            idx = int(input(f"Enter the index of the checkpoint to load: "))
            if idx < 0 or idx >= len(files):
                print("Index out of range.")
                idx = None
        except ValueError:
            print("Please enter a valid integer.")

    checkpoint_args = files[idx].split("_")[2:4]
    if [num_residual_blocks, num_hidden_channels] != [int(checkpoint_args[0]), int(checkpoint_args[1])]:
        print("The num_residual_blocks and num_hidden_channels args do not match!")
        return -1

    path = os.path.join(checkpoint_dir, files[idx])
    print(f"Loading checkpoint {files[idx]}...")
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    if not mode:
        optimizer.load_state_dict(ckpt['optim_state'])
        start_ep = ckpt.get('episode', 0)
        print(f"Resuming at episode {start_ep + 1}")
        return start_ep

In [5]:
class Node:
    """Represents a node in the Monte Carlo Search Tree."""
    def __init__(self, game, args, state, parent=None, action_taken=None, prior=0.0):
        self.game = game
        self.args = args
        self.state = state # ChessState object
        self.parent = parent
        self.action_taken = action_taken # Action index (int) that led here
        self.prior = prior # P(s,a) - prior probability of taking 'action_taken' from parent

        self.children = [] # List of child Nodes

        # Track expandable moves based on valid moves for this state's board
        # Calculate on demand or during expansion
        self._valid_moves_indices = None
        self._fully_expanded_check_done = False

        # MCTS statistics for the edge leading to this node (from parent)
        self.visit_count = 0 # N(s,a) where s=parent.state, a=action_taken
        self.total_action_value = 0.0 # W(s,a) - Sum of values from simulations passing through this edge, relative to player at parent

    @property
    def q_value(self):
        """Calculates the Q-value for the action leading to this node."""
        return self.total_action_value / self.visit_count if self.visit_count > 0 else 0.0

    def _get_valid_moves_indices(self):
        """Helper to get and cache valid move indices."""
        if self._valid_moves_indices is None:
            valid_moves_mask = self.game.get_valid_moves(self.state)
            self._valid_moves_indices = np.where(valid_moves_mask == 1)[0]
        return self._valid_moves_indices

    def is_fully_expanded(self):
        """Checks if all valid actions from this node have been expanded."""
        # A node is fully expanded if the number of children equals the number of valid moves
        if self._fully_expanded_check_done:
            return True # Optimization if we know it's done
        num_valid_moves = len(self._get_valid_moves_indices())
        is_full = len(self.children) >= num_valid_moves
        if is_full:
            self._fully_expanded_check_done = True # Cache result
        return is_full

    def select(self):
        """Selects the child with the highest UCB score."""
        best_child = None
        best_ucb = -np.inf

        parent_visit_count = sum(child.visit_count for child in self.children) # N(s) for current node state

        for child in self.children:
            # Pass W(s,a), N(s,a), N(s), P(s,a), C
            # Value W is stored from the perspective of the player at the parent node (self)
            ucb = _ucb_score(
                child_W=child.total_action_value,
                child_N=child.visit_count,
                parent_N=parent_visit_count, # Use sum of child visits as parent N(s)
                child_P=child.prior,
                C=self.args['C']
            )

            if ucb > best_ucb:
                best_ucb = ucb
                best_child = child

        return best_child

    def expand(self, policy_probs):
        """
        Creates child nodes for all valid actions using the provided policy.
        Assumes policy_probs is already masked for valid moves and normalized.
        """
        self._fully_expanded_check_done = False # Reset check status
        valid_action_indices = self._get_valid_moves_indices()

        for action_idx in valid_action_indices:
            # Check if child for this action already exists (e.g., if called partially before)
            if any(child.action_taken == action_idx for child in self.children):
                 continue

            # Create the next state object by applying the action
            next_state_obj = self.game.get_next_state(self.state, action_idx, player=None) # Player arg might not be needed
            prior_p = policy_probs[action_idx]

            # Create and add the child node
            child = Node(self.game, self.args, next_state_obj, parent=self, action_taken=action_idx, prior=prior_p)
            self.children.append(child)
        self._fully_expanded_check_done = True # Now fully expanded


    def backpropagate(self, value):
        """
        Updates statistics back up the tree to the root.
        Args:
            value: The value estimated for the current node (self), from the perspective
                   of the player whose turn it is AT THIS NODE'S STATE.
        """
        node = self
        # Value passed to parent needs to be from the perspective of the player at the parent node.
        # Since turns alternate, this means flipping the sign.
        current_value = value

        while node.parent is not None: # Stop before processing the absolute root's parent (None)
            # Update stats for the EDGE leading from parent to node
            # Visit count N(s,a) is stored in the child node (self)
            node.visit_count += 1
            # W(s,a) needs the value from the correct perspective (player at parent)
            # Flip value before adding to parent's edge W
            current_value = -current_value
            node.total_action_value += current_value

            # Move to the parent
            node = node.parent

        # Root node's visit count also needs update (represents N(root_state))
        # This happens implicitly as it's the 'node' in the last step before parent becomes None? Check logic.
        # Let's ensure root visit count is incremented. The loop stops when node.parent is None.
        # The final 'node' is the root. Its W doesn't matter as much, but N does.
        node.visit_count += 1 # Increment N(s) for the root state

In [6]:
# --- Self-Play Game State Holder ---

class SPG:
    def __init__(self, game):
        self.state = game.get_initial_state()
        self.memory = []
        self.root = None
        self.node = None

# --- Parallel MCTS Search ---

class MCTSParallel:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model
        self.device = next(model.parameters()).device

    @torch.no_grad()
    def search(self, states, spGames):
        # Ensure states are correctly handled (e.g., they contain a board object)
        encoded_states_list = [self.game.get_encoded_state(s) for s in states]
        encoded_batch = np.stack(encoded_states_list)
        encoded_batch_tensor = torch.tensor(encoded_batch, dtype=torch.float32, device=self.device)

        policy_logits, _ = self.model(encoded_batch_tensor)
        policy_batch = torch.softmax(policy_logits, dim=1).cpu().numpy()

        # Apply Dirichlet noise only if epsilon > 0
        noise = np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size, size=policy_batch.shape[0])
        policy_batch = (1 - self.args['dirichlet_epsilon']) * policy_batch + self.args['dirichlet_epsilon'] * noise

        for i, spg in enumerate(spGames):
            root_policy = policy_batch[i]
            current_state = states[i]
            valid_moves = self.game.get_valid_moves(current_state) # Should return mask
            root_policy *= valid_moves # Apply mask
            policy_sum = np.sum(root_policy)
            if policy_sum > 1e-8:
                root_policy /= policy_sum # Normalize
            else:
                # Fallback if policy sum is zero after masking
                num_valid = np.sum(valid_moves)
                if num_valid > 0: root_policy = valid_moves / num_valid
                else: root_policy = np.zeros(self.game.action_size, dtype=np.float32) # Should not happen if game not ended

            spg.root = Node(self.game, self.args, current_state, parent=None, action_taken=None, prior=0)
            spg.root.expand(root_policy)

        for _ in range(self.args['num_searches']):
            # Selection phase for each game
            for spg in spGames:
                spg.node = None # Reset expansion marker
                node = spg.root
                while node.is_fully_expanded():
                    if not node.children: break # Safety break
                    node = node.select()
                    if node is None: # Handle potential error in selection
                        node = spg.root
                        break

                value, is_terminal = self.game.get_value_and_terminated(node.state)

                if is_terminal:
                    node.backpropagate(value) # Backpropagate terminal value
                else:
                    # If not terminal and not expanded (leaf node), mark for evaluation
                    if not node.children:
                        spg.node = node # Mark this node in this SPG for expansion

            # Batch Expansion/Evaluation Phase
            expandable_indices = [idx for idx, spg in enumerate(spGames) if spg.node is not None]

            if len(expandable_indices) > 0:
                states_to_evaluate = [spGames[idx].node.state for idx in expandable_indices]
                encoded_batch_list = [self.game.get_encoded_state(s) for s in states_to_evaluate]
                encoded_batch = np.stack(encoded_batch_list)
                encoded_batch_tensor = torch.tensor(encoded_batch, dtype=torch.float32, device=self.device)

                policy_logits, value_batch_tensor = self.model(encoded_batch_tensor)
                policy_batch = torch.softmax(policy_logits, dim=1).cpu().numpy()
                value_batch = value_batch_tensor.cpu().numpy().flatten()

                for i, game_idx in enumerate(expandable_indices):
                    node_to_expand = spGames[game_idx].node
                    nn_policy = policy_batch[i]
                    nn_value = value_batch[i] # Value relative to player turn at node_to_expand.state

                    valid_moves = self.game.get_valid_moves(node_to_expand.state)
                    nn_policy *= valid_moves
                    policy_sum = np.sum(nn_policy)
                    if policy_sum > 1e-8:
                        nn_policy /= policy_sum
                    else:
                        num_valid = np.sum(valid_moves)
                        if num_valid > 0: nn_policy = valid_moves / num_valid
                        else: nn_policy = np.zeros(self.game.action_size, dtype=np.float32)

                    node_to_expand.expand(nn_policy)
                    node_to_expand.backpropagate(nn_value) # Backpropagate NN value


# --- AlphaZero Agent with Parallel Self-Play ---

class AlphaZeroAgentParallel:
    def __init__(self, game, model, optimizer, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game # Expects game object with methods like get_initial_state, etc.
        self.args = args
        self.mcts = MCTSParallel(game, args, model)
        # Use deque for efficient memory buffer
        self.memory = collections.deque(maxlen=args.get('replay_buffer_size', 50000))

    def display_active_game_states(self, spGames, active_indices, turn_count):
        os.system('cls' if os.name == 'nt' else 'clear')
        print(f"\n--- Active Game States (Turn approx {turn_count}) ---")
        # Default to displaying up to 4 games based on user comment
        max_display = self.args.get('vis_max_games_display', 4)
        active_indices.sort()

        board_outputs = []
        for i, game_idx in enumerate(active_indices):
            if i >= max_display:
                 board_outputs.append(f"... and {len(active_indices) - max_display} more active games.")
                 break
            spg = spGames[game_idx]
            header = f"    --- Game {game_idx + 1} ---"
            board_str = spg.state.board.unicode()
            board_outputs.append(header + "\n" + board_str)

        # Print boards sequentially
        for output in board_outputs[:-1]:
             print(output)
             print("-" * 22) # Separator

        print(board_outputs[-1])
        print("=" * 94) # Footer line


    def selfPlay(self, display_interval):
        return_memory = []
        num_parallel = self.args['num_parallel_games']
        spGames = [SPG(self.game) for _ in range(num_parallel)]
        active_game_indices = list(range(num_parallel))
        turn_count = 0
        # Store Zobrist hash history
        game_state_histories = {idx: [] for idx in active_game_indices}
        for idx in active_game_indices:
             try:
                 # Ensure initial state has a board attribute for hashing
                 if hasattr(spGames[idx].state, 'board'):
                    game_state_histories[idx].append(chess.polyglot.zobrist_hash(spGames[idx].state.board))
             except Exception as e:
                 print(f"Warning: Could not hash initial board for game {idx}: {e}")


        while len(active_game_indices) > 0:
            start_turn_time = time.time()
            current_states = [spGames[i].state for i in active_game_indices]
            active_spGames = [spGames[i] for i in active_game_indices]

            self.mcts.search(current_states, active_spGames)

            new_active_indices = []
            finished_this_step_info = []

            for list_idx, game_idx in enumerate(active_game_indices):
                spg = spGames[game_idx]
                if spg.root is None:
                    finished_this_step_info.append({'id': game_idx, 'outcome': 'Error - No Root', 'length': len(game_state_histories.get(game_idx, []))})
                    continue

                action_probs = np.zeros(self.game.action_size, dtype=np.float32)
                root_total_visits = sum(child.visit_count for child in spg.root.children)
                if root_total_visits > 0:
                    for child in spg.root.children:
                        if child.action_taken is not None: action_probs[child.action_taken] = child.visit_count / root_total_visits
                else:
                    valid_m = self.game.get_valid_moves(spg.state)
                    num_v = np.sum(valid_m)
                    if num_v > 0: action_probs = valid_m / num_v
                    else:
                        value, _ = self.game.get_value_and_terminated(spg.state)
                        finished_this_step_info.append({'id': game_idx, 'outcome': value, 'length': len(game_state_histories.get(game_idx, []))})
                        continue

                encoded_state = self.game.get_encoded_state(spg.state)
                spg.memory.append((encoded_state, action_probs))

                temp = self.args['temperature']
                prob_sum = np.sum(action_probs)
                if prob_sum < 1e-6:
                     value, _ = self.game.get_value_and_terminated(spg.state)
                     finished_this_step_info.append({'id': game_idx, 'outcome': value, 'length': len(game_state_histories.get(game_idx, []))})
                     continue
                # Normalize probabilities before applying temperature or sampling
                if abs(prob_sum - 1.0) > 1e-5: action_probs /= prob_sum

                action_idx = -1
                try:
                     if temp < 1e-3:
                         # Argmax: ensure we only consider valid moves' indices if probs are zero elsewhere
                         valid_indices = np.where(self.game.get_valid_moves(spg.state) == 1)[0]
                         if len(valid_indices) > 0:
                             action_idx = valid_indices[np.argmax(action_probs[valid_indices])]
                         else: # No valid moves, game should end
                             raise ValueError("No valid moves for argmax selection.")
                     else:
                         # Sample using temperature
                         temp_probs = action_probs ** (1.0 / temp)
                         temp_probs_sum = np.sum(temp_probs)
                         if temp_probs_sum < 1e-8: # Avoid division by zero if temp makes probs tiny
                              raise ValueError("Probability sum near zero after temperature.")
                         temp_probs /= temp_probs_sum
                         # Ensure probabilities are still valid after potential float issues
                         if abs(np.sum(temp_probs) - 1.0) > 1e-5 or np.any(np.isnan(temp_probs)):
                              raise ValueError("Invalid probabilities after temperature normalization.")
                         action_idx = np.random.choice(self.game.action_size, p=temp_probs)
                except (ValueError, FloatingPointError, IndexError) as e:
                     # Fallback if sampling fails or no valid moves
                     valid_m = self.game.get_valid_moves(spg.state)
                     num_v = np.sum(valid_m)
                     if num_v > 0: # Choose a random valid move as fallback
                         action_idx = np.random.choice(np.where(valid_m == 1)[0])
                     else: # No valid moves, game must be over
                         value, _ = self.game.get_value_and_terminated(spg.state)
                         finished_this_step_info.append({'id': game_idx, 'outcome': value, 'length': len(game_state_histories.get(game_idx, []))})
                         continue # Skip state update

                if action_idx < 0 :
                     value, _ = self.game.get_value_and_terminated(spg.state)
                     finished_this_step_info.append({'id': game_idx, 'outcome': value, 'length': len(game_state_histories.get(game_idx, []))})
                     continue

                spg.state = self.game.get_next_state(spg.state, action_idx, player=None)

                try: # Store Zobrist hash
                    if hasattr(spg.state, 'board'):
                        game_state_histories[game_idx].append(chess.polyglot.zobrist_hash(spg.state.board))
                except Exception as e:
                     print(f"Warning: Could not hash board for game {game_idx} after move: {e}")


                value, is_terminal = self.game.get_value_and_terminated(spg.state)

                if is_terminal:
                    outcome = value
                    final_len = len(game_state_histories.get(game_idx, []))
                    # Assign rewards alternating by turn from end state
                    for i in range(len(spg.memory)):
                         hist_encoded_state, hist_action_probs = spg.memory[i]
                         reward = outcome if (len(spg.memory) - 1 - i) % 2 == 0 else -outcome
                         return_memory.append((hist_encoded_state, hist_action_probs, reward))
                    finished_this_step_info.append({'id': game_idx, 'outcome': outcome, 'length': final_len})
                else:
                    new_active_indices.append(game_idx)

            active_game_indices = new_active_indices
            turn_count += 1

            if display_interval > 0 and turn_count % display_interval == 0 and len(active_game_indices) > 0:
                end_turn_time = time.time()
                turn_duration = end_turn_time - start_turn_time
                self.display_active_game_states(spGames, active_game_indices, turn_count)
                if finished_this_step_info:
                     print("Games finished this turn:")
                     for info in finished_this_step_info:
                          print(f"  Game {info['id']}: Outcome {info.get('outcome', 'N/A')}, {info.get('length', 'N/A')} moves")
                print(f"Turn took {turn_duration:.3f}s. Active games: {len(active_game_indices)}")

        return return_memory, game_state_histories


    def train(self, memory_batch):
        # Ensure model is on the correct device before converting batch
        self.model.to(self.model.device)

        state, policy_targets, value_targets = zip(*memory_batch)
        # Ensure state elements are numpy arrays before stacking
        if not isinstance(state[0], np.ndarray):
             # Assuming state is list of objects needing encoding
             state = [self.game.get_encoded_state(s) for s in state]

        state = np.array(state, dtype=np.float32)
        policy_targets = np.array(policy_targets, dtype=np.float32)
        value_targets = np.array(value_targets, dtype=np.float32).reshape(-1, 1)

        state_tensor = torch.tensor(state, dtype=torch.float32, device=self.model.device)
        policy_targets_tensor = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
        value_targets_tensor = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)

        out_policy_logits, out_value = self.model(state_tensor)

        policy_loss = -torch.sum(policy_targets_tensor * F.log_softmax(out_policy_logits, dim=1), dim=1).mean()
        value_loss = F.mse_loss(out_value, value_targets_tensor)
        loss = policy_loss + value_loss

        self.optimizer.zero_grad()
        loss.backward()
        # Optional gradient clipping
        # torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        self.optimizer.step()
        return loss.item(), policy_loss.item(), value_loss.item()

    def learn(self, start_iteration):
        args = self.args
        display_interval = args.get('display_interval', 0)

        for iteration in range(start_iteration, args['num_iterations']):
            iteration_start_time = time.time()
            print(f"\n===== Iteration {iteration + 1}/{args['num_iterations']} =====")

            self.model.eval()
            print(f"--- Starting Self-Play ---")
            iteration_memory = []
            games_generated_this_iter = 0
            target_games = args['num_selfPlay_iterations']
            run_num = 0
            all_histories_this_iter = {} # Store all histories if needed

            # Loop to generate the target number of games, running batches in parallel
            while games_generated_this_iter < target_games:
                 run_num += 1
                 # Determine how many games this batch contributes (it's always num_parallel_games)
                 games_in_batch = args['num_parallel_games']
                 print(f"  Self-play run {run_num}: Running {games_in_batch} games (Target: {target_games}, Total Generated Will Be: {games_generated_this_iter + games_in_batch})")

                 run_memory, run_histories = self.selfPlay(display_interval=display_interval)
                 iteration_memory.extend(run_memory)
                 all_histories_this_iter.update(run_histories) # Store/update histories
                 games_generated_this_iter += games_in_batch # Update count based on batch size

            self.memory.extend(iteration_memory) # Add collected data to main buffer
            self_play_end_time = time.time()
            print(f"\nSelf-Play finished for iteration. Generated {games_generated_this_iter} games ({len(iteration_memory)} samples) in {self_play_end_time - iteration_start_time:.2f}s. Buffer size: {len(self.memory)}")

            if len(self.memory) < args['batch_size']:
                 print("Not enough memory to train yet ({}/{} required). Skipping training phase.".format(len(self.memory), args['batch_size']))
                 continue

            self.model.train()
            print("--- Starting Training ---")
            total_iter_loss, total_iter_pol_loss, total_iter_val_loss = 0, 0, 0
            num_batches_trained = 0

            for epoch in range(args['num_epochs']):
                 print(f"  Epoch {epoch + 1}/{args['num_epochs']}")
                 num_samples = len(self.memory)
                 # Calculate steps based on current memory size for this epoch
                 num_steps_per_epoch = num_samples // args['batch_size']
                 if num_steps_per_epoch == 0:
                     print(f"    Not enough samples ({num_samples}) for a full batch size ({args['batch_size']}). Skipping epoch.")
                     continue

                 step_range = range(num_steps_per_epoch)
                 # Optional tqdm integration:
                 # try:
                 #     from tqdm import trange
                 #     step_range = trange(num_steps_per_epoch, desc=f"Epoch {epoch+1} Training", leave=False)
                 # except ImportError: pass

                 for step in step_range:
                     batch = random.sample(self.memory, args['batch_size'])
                     loss, pol_loss, val_loss = self.train(batch)
                     total_iter_loss += loss
                     total_iter_pol_loss += pol_loss
                     total_iter_val_loss += val_loss
                     num_batches_trained += 1
                     # Optional tqdm update:
                     # if 'trange' in globals() and isinstance(step_range, trange):
                     #      step_range.set_postfix(loss=f"{loss:.4f}")

            if num_batches_trained > 0:
                print(f"Training finished. Avg Loss over {num_batches_trained} batches: {total_iter_loss / num_batches_trained:.4f} (Policy: {total_iter_pol_loss / num_batches_trained:.4f}, Value: {total_iter_val_loss / num_batches_trained:.4f})")
            else:
                 print("No training steps performed (likely due to insufficient memory).")

            # --- Model Saving Logic ---
            if (iteration + 1) % args.get('save_frequency', 5) == 0:
                print(f"Saving model and optimizer state for iteration {iteration + 1}...")
                try:
                    checkpoint_path = f"{CHECKPOINT_DIR}/model_{iteration + 1}_{self.args['num_residual_blocks']}_{self.args['num_hidden_channels']}_parallel_chess.pt"
                    save_checkpoint(self.model, self.optimizer, iteration + 1, checkpoint_path)
                except Exception as e:
                    print(f"Error saving checkpoint: {e}")

In [7]:
###########################################
# Environment: ChessGame, ChessState, Encoder
###########################################
class ChessGame:
    """
    Represents the chess game environment.
    Handles game state transitions, move validation and state encoding.
    """
    def __init__(self):
        self.row_count = 8
        self.column_count = 8
        self.encoder = ChessEncoder()
        self.action_size = len(self.encoder.index_to_move)

    def get_initial_state(self):
        """Returns the initial chess state."""
        return ChessState()

    def get_valid_moves(self, state):
        """
        Returns a binary array (length=action_size) indicating legal moves
        based on the current board state.
        """
        valid_moves = np.zeros(self.action_size, dtype=np.uint8)
        legal_moves = [move.uci() for move in state.board.legal_moves]
        for move in legal_moves:
            idx = self.encoder.move_to_policy_index(move)
            if idx is not None:
                valid_moves[idx] = 1
        return valid_moves

    def get_next_state(self, state, action_index, player):
        """
        Given the current state and an action (by its index), returns the next state.
        The move is decoded using the encoder.
        """
        next_state = state.copy()
        move_str = self.encoder.policy_index_to_move(action_index)
        if move_str is None:
            return next_state
        chess_move = chess.Move.from_uci(move_str)
        if chess_move in next_state.board.legal_moves:
            next_state.board.push(chess_move)
        return next_state

    def get_reward(self, state):
        """
        Returns the reward for the given state.
        For a checkmate, returns 1 (win) or -1 (loss);
        For draws, returns 0; otherwise, returns None (non-terminal).
        """
        if state.board.is_checkmate():
            # If checkmate occurs, the side who just moved wins.
            return -1 if state.board.turn else 1
        if (state.board.is_stalemate() or state.board.is_insufficient_material() or
                state.board.is_fifty_moves() or state.board.is_repetition()):
            return 0
        return None  # Game not terminated

    def get_encoded_state(self, state):
        """
        Encodes the board into a tensor suitable for neural network input.
        We use a 14-channel representation:
          - Channels 0..11: One-hot planes for each piece type (6 for white, 6 for black)
          - Channel 12: Turn indicator (1 if white to move, 0 if black)
          - Channel 13: Castling rights indicator (simplified)
        Returns data with shape (channels, 8, 8).
        """
        piece_map_dict = state.board.piece_map()
        num_pieces = len(piece_map_dict)
        squares = np.empty(num_pieces, dtype=np.int32)
        symbols_ord = np.empty(num_pieces, dtype=np.int32) # Use ordinals
        idx = 0
        for square, piece in piece_map_dict.items():
            squares[idx] = square
            symbols_ord[idx] = ord(piece.symbol()) # Store unicode ordinal
            idx += 1

        turn = state.board.turn == chess.WHITE
        castling = (state.board.has_kingside_castling_rights(chess.WHITE) or
                    state.board.has_queenside_castling_rights(chess.WHITE) or
                    state.board.has_kingside_castling_rights(chess.BLACK) or
                    state.board.has_queenside_castling_rights(chess.BLACK))
        encoded_array = _encode_state_numba(squares, symbols_ord, turn, castling)
        return encoded_array

    def board_to_array(self, board):
        """Converts a board to a simple representation (e.g., piece IDs)."""
        piece_ids = np.full(64, -1, dtype=np.int32) # Use -1 for empty
        mapping = {'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,
                   'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11}
        for sq, piece in board.piece_map().items():
            piece_ids[sq] = mapping.get(piece.symbol(), -1) # Use get for safety
        return piece_ids

    def get_value_and_terminated(self, state, action_taken=None): # action_taken might not be needed if checking current state
        """
        Checks if the current state is terminal.
        Returns (value, is_terminal). Value is from the perspective of the
        player whose turn it IS at the current state.
        """
        board = state.board # Assuming state is a ChessState object
        outcome = board.outcome() # Checks for checkmate, stalemate, rules, etc.

        if outcome is not None:
            # Game has ended
            is_terminal = True
            if outcome.winner == chess.WHITE:
                 # White won. Value is +1 if White's turn (Black just moved), -1 if Black's turn (White just moved).
                 value = 1.0 if board.turn == chess.WHITE else -1.0
            elif outcome.winner == chess.BLACK:
                 # Black won. Value is -1 if White's turn (Black just moved), +1 if Black's turn (White just moved).
                 value = -1.0 if board.turn == chess.WHITE else 1.0
            else: # Draw
                 value = 0.0
            return value, is_terminal
        else:
            # Game not ended
            return 0.0, False

    def get_opponent(self, player):
        """Returns the opponent player identifier (if using 1/-1)."""
        # If using player = 1 for white, -1 for black
        return -player
        # If using player = chess.WHITE, chess.BLACK
        # return chess.BLACK if player == chess.WHITE else chess.WHITE

    def get_opponent_value(self, value):
        """Returns the value from the opponent's perspective."""
        return -value

class ChessState:
    """
    Wrapper for a chess.Board. Provides a copy method for state cloning.
    """
    def __init__(self, board=None):
        self.board = board if board is not None else chess.Board()

    def copy(self):
        return ChessState(self.board.copy())

class ChessEncoder:
    """
    Encodes and decodes chess moves into policy indices.
    Maintains a mapping between UCI move strings and indices.
    """
    def __init__(self):
        self.move_to_index = {}
        self.index_to_move = []
        self._create_move_mappings()

    def _create_move_mappings(self):
        """Creates mappings for all possible moves including underpromotions."""
        files = 'abcdefgh'
        ranks = '12345678'
        pieces = ['q', 'r', 'b', 'n']

        for f1 in files:
            for r1 in ranks:
                for f2 in files:
                    for r2 in ranks:
                        # Skip null moves (e.g., "a1a1")
                        if f1 == f2 and r1 == r2:
                            continue
                        move = f"{f1}{r1}{f2}{r2}"
                        self._add_move(move)
                        # Underpromotions for white (7->8) and black (2->1)
                        if r1 == '7' and r2 == '8':
                            for p in pieces:
                                self._add_move(f"{f1}{r1}{f2}{r2}{p}")
                        if r1 == '2' and r2 == '1':
                            for p in pieces:
                                self._add_move(f"{f1}{r1}{f2}{r2}{p}")

    def _add_move(self, move):
        if move not in self.move_to_index:
            self.move_to_index[move] = len(self.index_to_move)
            self.index_to_move.append(move)

    def move_to_policy_index(self, move_uci):
        return self.move_to_index.get(move_uci)

    def policy_index_to_move(self, index):
        if 0 <= index < len(self.index_to_move):
            return self.index_to_move[index]
        return None

###########################################
# Neural Network Definitions
###########################################
class NeuralNetwork(nn.Module):
    """
    Abstract neural network for policy and value predictions.
    """
    def __init__(self):
        super().__init__()

    def forward(self, x):
        raise NotImplementedError("Subclasses must implement this method.")

class ResidualBlock(nn.Module):
    """
    Standard Residual Block with two 3x3 convolutions.
    """
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        return F.relu(x)

class ResNetAlphaZero(NeuralNetwork):
    """
    ResNet architecture for policy and value predictions.
    Consists of an initial convolution, a backbone of residual blocks,
    a policy head, and a value head.
    """
    def __init__(self, game, num_residual_blocks, num_hidden_channels, device):
        super().__init__()
        self.device = device

        # Initial convolution block; input channels = 14 (per get_encoded_state)
        self.conv1 = nn.Conv2d(14, num_hidden_channels, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(num_hidden_channels)

        # Residual blocks (backbone)
        self.res_blocks = nn.ModuleList(
            [ResidualBlock(num_hidden_channels) for _ in range(num_residual_blocks)]
        )

        # Policy head: outputs logits for every possible action.
        self.policy_head = nn.Sequential(
            nn.Conv2d(num_hidden_channels, 2, kernel_size=1),
            nn.BatchNorm2d(2),
            nn.Flatten(),
            nn.Linear(2 * 8 * 8, game.action_size)
        )

        # Value head: outputs a single scalar evaluation.
        self.value_head = nn.Sequential(
            nn.Conv2d(num_hidden_channels, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Flatten(),
            nn.Linear(1 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Tanh()
        )
        self.to(device)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        for block in self.res_blocks:
            x = block(x)
        policy = self.policy_head(x)
        value = self.value_head(x)
        # Return softmax probabilities and the value prediction.
        return F.softmax(policy, dim=1), value

###########################################
# Monte Carlo Tree Search (MCTS)
###########################################
class MCTS:
    """
    Implements Monte Carlo Tree Search (MCTS) using optimized state keys.
    Instead of full FEN strings, we use a Zobrist hash for fast lookups.
    """
    def __init__(self, game, model, args):
        self.game = game
        self.model = model
        self.args = args
        self.Qsa = {}  # Q-values for (state_key, action)
        self.Nsa = {}  # Visit counts for (state_key, action)
        self.Ns = {}   # Visit counts for states (indexed by state_key)
        self.Ps = {}   # Prior probabilities for states, from the network (indexed by state_key)

    def _state_key(self, board):
        # Compute a 64-bit integer hash for the board using a Zobrist hash.
        return chess.polyglot.zobrist_hash(board)

    def run(self, initial_state, temp=1.0):
        """
        Runs a fixed number of MCTS simulations and returns action probabilities.
        """
        board = initial_state.board
        key = self._state_key(board)
        valid = self.game.get_valid_moves(initial_state) # Need valid moves mask early

        # Ensure root node is initialized and noise applied
        if key not in self.Ps:
             encoded = self.game.get_encoded_state(initial_state)
             tensor = torch.tensor(encoded).unsqueeze(0).float().to(self.model.device)
             # Get raw logits from the model IF it returns them
             # Assuming self.model already returns softmax probabilities based on your code:
             policy_probs, _ = self.model(tensor)
             priors = policy_probs.cpu().detach().numpy()[0]

             # Apply dirichlet noise using Numba function
             self.Ps[key] = _apply_dirichlet_noise(priors, valid,
                                                 self.args['dirichlet_alpha'],
                                                 self.args['dirichlet_epsilon'])
             self.Ns[key] = 0

        # Run simulations
        for _ in range(self.args['num_searches']):
            self._search(board) # Modifies board in place

        # Calculate final probabilities using Numba function
        counts = np.array([self.Nsa.get((key, a), 0) for a in range(self.game.action_size)], dtype=np.float32) # Use float32
        probs = _calculate_final_probs(counts, valid, temp)
        return probs

    def _search(self, board):
        """
        Performs one MCTS simulation (Selection, Expansion/Evaluation, Backpropagation)
        using optimized keys.
        """
        path = []
        moves_made = []
        current_board = board # Use local var

        while True:
            # Check terminal state
            state_obj = ChessState(board=current_board) # Temp object for checks/encoding
            outcome = current_board.outcome() # More robust way to check termination

            if outcome is not None:
                # Game has ended (checkmate, stalemate, draw rule, etc.)
                if outcome.winner == chess.WHITE:
                    value = 1.0 # Correct for the player who made the winning move
                elif outcome.winner == chess.BLACK:
                    value = 1.0 # Correct for the player who made the winning move
                else: # Draw
                    value = 0.0
                # The sign flip during backpropagation handles the perspective shift.
                break # Break from the while loop to start backpropagation

            key = self._state_key(current_board)

            # Expansion / Evaluation (If node not visited before)
            if key not in self.Ps:
                encoded = self.game.get_encoded_state(state_obj)
                tensor = torch.tensor(encoded).unsqueeze(0).float().to(self.model.device)
                policy_probs, value_tensor = self.model(tensor) # Assuming softmax output
                pi = policy_probs.cpu().detach().numpy()[0]

                valid = self.game.get_valid_moves(state_obj)
                pi *= valid # Mask invalid moves

                sum_pi = np.sum(pi)
                if sum_pi > 1e-8: # Renormalize
                    pi /= sum_pi
                else: # Handle case where all valid moves have 0 prior
                    num_valid = np.sum(valid)
                    if num_valid > 0: pi = valid / num_valid
                    # else: pi remains zero, game should have ended if no valid moves

                self.Ps[key] = pi
                self.Ns[key] = 0 # Initialize visit count for the new node

                value = -value_tensor.item()
                break  # Leaf node reached, break to start backpropagation

            # --- Selection using efficient Numba function ---
            else: # Node has been visited before (key is in Ps)
                valid_moves_mask = self.game.get_valid_moves(state_obj) # Get valid moves for selection
                action_size = self.game.action_size

                # Prepare arrays needed by select_ucb_array
                Q_vals_arr = np.array([self.Qsa.get((key, a), 0.0) for a in range(action_size)], dtype=np.float32)
                N_sa_arr = np.array([self.Nsa.get((key, a), 0) for a in range(action_size)], dtype=np.int32)
                P_prior_arr = self.Ps[key]  # Fetch the stored numpy array for priors
                N_s_val = self.Ns[key]      # Fetch the integer visit count for state

                action = select_ucb_array(Q_vals_arr, N_sa_arr, N_s_val, P_prior_arr,
                                          self.args['C'], valid_moves_mask)

                if action == -1:
                    print(f"ERROR: Efficient selection failed for key {key}. Board:\n{current_board}")
                    print(f"Valid mask sum: {np.sum(valid_moves_mask)}")
                    # Handle error state, e.g., assign draw value and break
                    value = 0.0
                    break

                # Store state_key and selected action for backpropagation
                path.append((key, action))

                # Make the move on the board
                move_uci = self.game.encoder.policy_index_to_move(action)
                if move_uci is None:
                    print(f"ERROR: Invalid action index {action} selected by efficient selection.")
                    value = 0.0; break # Assign draw value and break

                try:
                    move = chess.Move.from_uci(move_uci)
                    if move not in current_board.legal_moves:
                         # Should not happen if valid_moves_mask is correct!
                         print(f"ERROR: Efficient MCTS selected illegal move {move_uci} for board:\n{current_board}")
                         value = 0.0; break # Assign draw value and break

                    current_board.push(move)
                    moves_made.append(move) # Keep track of moves to pop later
                except ValueError:
                    print(f"ERROR: Could not parse move UCI {move_uci} from action index {action}.")
                    value = 0.0; break # Assign draw value and break

        # Backpropagation (Value is from perspective of player who made the *last move* in the path)
        for state_key, action in reversed(path):
            n_sa = self.Nsa.get((state_key, action), 0) + 1
            self.Nsa[(state_key, action)] = n_sa
            prev_q = self.Qsa.get((state_key, action), 0.0)
            # Q_new = ( Q_old * (N_sa - 1) + value ) / N_sa
            self.Qsa[(state_key, action)] = ((prev_q * (n_sa - 1) + value) / n_sa)

            self.Ns[state_key] = self.Ns.get(state_key, 0) + 1 # Increment state visit count

            # Flip value for the parent node (representing opponent's perspective)
            value = -value

        # UNDO all the moves made during this simulation
        for _ in moves_made:
            current_board.pop()

###########################################
# AlphaZero Agent: Self-play and Training
###########################################
class AlphaZeroAgent:
    """
    Orchestrates self-play, MCTS move selection, and training updates.
    """
    def __init__(self, game, model, optimizer, args):
        self.game = game
        self.model = model
        self.optimizer = optimizer
        self.args = args
        self.mcts = MCTS(game, model, args)
        self.memory = collections.deque(maxlen=args['replay_buffer_size'])  # Training examples: (encoded_state, action_probs, outcome)

    def self_play(self):
        """
        Runs one self-play episode until termination.
        """
        memory = []
        state = self.game.get_initial_state()
        player = 1
        move_num = 1

        while True:
            print(f"\nMove {move_num}, Player: {player}")
            print(state.board.unicode())
            # Use a higher temperature (more exploration) early in the game.
            temp = 1.0 if move_num <= self.args['temperature_threshold'] else 0.0
            action_probs = self.mcts.run(state, temp=temp)
            encoded = self.game.get_encoded_state(state)
            memory.append((encoded, action_probs))

            # Choose action based on probabilities
            # Ensure probabilities sum to 1, handle potential float errors
            action_probs_sum = np.sum(action_probs)
            if abs(action_probs_sum - 1.0) > 1e-5:
                 print(f"Warning: Action probabilities do not sum to 1 ({action_probs_sum}). Renormalizing.")
                 action_probs /= action_probs_sum # Renormalize

            action_index = np.random.choice(self.game.action_size, p=action_probs)
            move = self.game.encoder.policy_index_to_move(action_index)
            print(f"Chosen move index: {action_index}, Move: {move}")

            state = self.game.get_next_state(state, action_index, player)
            move_num += 1

            outcome = state.board.outcome()
            if outcome is not None:
                # Game ended
                if outcome.winner == chess.WHITE:
                    final_reward = 1.0
                elif outcome.winner == chess.BLACK:
                    final_reward = -1.0
                else: # Draw
                    final_reward = 0.0

                print(f"\nGame terminated. Result: {outcome.termination.name}, Winner: {outcome.winner}, Final Reward perspective White: {final_reward}")
                print("Final board:")
                print(state.board.unicode())

                # --- Add experience to the deque from the correct perspectives ---
                # Iterate through the moves made in the game stored in current_game_memory
                player_turn = 1 # Start from white's perspective for the first stored state
                for i, (enc_state, act_probs) in enumerate(memory):
                    # The reward is assigned relative to the player whose turn it was when the state was recorded.
                    # If final_reward is 1 (White won), states recorded during White's turn get +1, Black's turn get -1.
                    # If final_reward is -1 (Black won), states recorded during White's turn get -1, Black's turn get +1.
                    # If final_reward is 0 (Draw), all states get 0.
                    perspective_reward = final_reward * player_turn
                    self.memory.append((enc_state, act_probs, perspective_reward))
                    player_turn *= -1 # Flip perspective for the next state recorded

                break # Exit the while loop for this episode

            # --- Switch player perspective for the next iteration ---
            # This perspective is only for keeping track internally, the board object handles turns.
            player *= -1
            # Reset the MCTS tree for each move.
            self.mcts = MCTS(self.game, self.model, self.args)

    def train(self, batch):
        """
        Performs one training update on a given batch.
        """
        states, policy_targets, value_targets = zip(*batch)
        states = torch.tensor(np.array(states, dtype=np.float32)).float().to(self.model.device)
        policy_targets = torch.tensor(np.array(policy_targets, dtype=np.float32)).float().to(self.model.device)
        value_targets = torch.tensor(np.array(value_targets, dtype=np.float32)).float().unsqueeze(1).to(self.model.device)

        self.optimizer.zero_grad()
        policy_preds, value_preds = self.model(states)
        policy_loss = -torch.mean(torch.sum(policy_targets * torch.log(policy_preds + 1e-8), dim=1))
        value_loss = F.mse_loss(value_preds, value_targets)
        total_loss = policy_loss + value_loss
        total_loss.backward()
        self.optimizer.step()
        return total_loss.item(), policy_loss.item(), value_loss.item()

    def learn(self, start_iteration=0):
        """
        Main training loop: run self-play to collect experience and update the network.
        """
        for episode in range(start_iteration, self.args['num_episodes']):
            print(f"\n=== Starting episode {episode + 1}/{self.args['num_episodes']} ===")
            # Set model to evaluation mode during self-play for efficiency
            # (disables dropout, uses running averages for batch norm)
            self.model.eval()
            self.self_play()
            print(f"Episode {episode + 1} finished; Replay buffer size: {len(self.memory)}/{self.args['replay_buffer_size']}")

            if len(self.memory) >= self.args['batch_size']:
                self.model.train()
                batch = random.sample(self.memory, self.args['batch_size'])
                loss, pol_loss, val_loss = self.train(batch)
                print(f"Training loss: {loss:.4f} (Policy: {pol_loss:.4f}, Value: {val_loss:.4f})")

            if (episode + 1) % self.args['save_frequency'] == 0:
                print(f"Saving model checkpoint at episode {episode + 1}...")
                checkpoint_path = f"{CHECKPOINT_DIR}/model_{episode + 1}_{self.args['num_residual_blocks']}_{self.args['num_hidden_channels']}_chess.pt"
                save_checkpoint(self.model, self.optimizer, episode + 1, checkpoint_path)

In [8]:
###########################################
# Main Function: Setup and Training
###########################################
def main():
    # Hyperparameters inspired by the AlphaZero chess paper:
    args = {
        'num_residual_blocks': 20,           # 20 residual blocks (as used in the paper)
        'num_hidden_channels': 256,          # 256 filters per layer
        'num_searches': 400,                # 800 MCTS simulations per move (adjust based on hardware)
        'C': 2,                            # Exploration constant (C_puct)
        'num_episodes': 1000,                # Number of self-play episodes
        'temperature': 1.25,                  # Exploration temperature
        'num_iterations': 10,               # Total training iterations
        'num_selfPlay_iterations': 4,       # Target games per iteration
        'num_parallel_games': 4,         # Games per selfPlay batch run
        'num_epochs': 4,                    # Effectively replaced by training_steps_per_iteration
        'batch_size': 256,                   # Batch size for training (adjust based on GPU memory)
        'replay_buffer_size': 2000,         # --- ADDED: Max size of the replay buffer (adjust based on RAM) ---
        'learning_rate': 1e-3,              # Learning rate
        'temperature_threshold': 30,         # Number of plies (half-moves) for temperature=1 exploration
        'save_frequency': 1,                # Save model every 50 episodes
        'dirichlet_epsilon': 0.25,           # Weight of Dirichlet noise in root priors
        'dirichlet_alpha': 0.3,               # Shape parameter for Dirichlet noise
        'vis_max_games_display': 1,        # Added default
        'display_interval': 100               # Added default (display every 5 turns)
    }

    game = ChessGame()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = ResNetAlphaZero(game, args['num_residual_blocks'], args['num_hidden_channels'], device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args['learning_rate'], weight_decay=1e-4)
    agent = AlphaZeroAgentParallel(game, model, optimizer, args)

    load_model = input("Train model from scratch (T) or from checkpoint (F): ")
    start_ep = choose_and_load_checkpoint(model, optimizer, num_residual_blocks=args['num_residual_blocks'], num_hidden_channels=args['num_hidden_channels'], device=device) if load_model == "F" else 0
    if start_ep != -1:
        agent.learn(start_iteration=start_ep)

if __name__ == "__main__":
    main()

Using device: cuda
Train model from scratch (T) or from checkpoint (F): T

===== Iteration 1/10 =====
--- Starting Self-Play ---
  Self-play run 1: Running 4 games (Target: 4, Total Generated Will Be: 4)


KeyboardInterrupt: 

In [ ]:
import chess.engine

def play_model_vs_stockfish(game, model, args,
                            stockfish_path: str,
                            stockfish_time: float = 0.1,
                            stockfish_skill_level: int = 20,
                            model_searches: int = 800,
                            stockfish_plays_white: bool = False):
    """
    Let your AlphaZero agent play against Stockfish.

    Args:
      game, model, args      — your existing classes & hyperparameters
      stockfish_path (str)   — path to the Stockfish binary
      stockfish_time (float) — thinking time per move (seconds)
      model_searches (int)   — MCTS sims for your model
      stockfish_plays_white (bool)
                              — if True, Stockfish moves first as White
    """
    # prepare MCTS for the model
    play_args = args.copy()
    play_args['num_searches'] = model_searches
    mcts = MCTS(game, model, play_args)

    # launch Stockfish engine
    engine = chess.engine.SimpleEngine.popen_uci(stockfish_path)
    engine.configure({"Threads": 2, "Skill Level": stockfish_skill_level})  # optional: tune to your CPU

    board = chess.Board()
    print("Starting a match: Stockfish as", "White" if stockfish_plays_white else "Black")

    move_num = 1
    while not board.is_game_over():
        print(board.unicode())

        # Stockfish to move?
        if board.turn == chess.WHITE and stockfish_plays_white or \
           board.turn == chess.BLACK and not stockfish_plays_white:
            # You can also call engine.go(depth=20) if you prefer fixed depth
            result = engine.play(board, chess.engine.Limit(time=stockfish_time))
            board.push(result.move)
            print(f"\nMove {move_num}")
            print(f"Stockfish plays {result.move}")
        else:
            # Model to move
            state = ChessState(board=board.copy())  # or however you track history
            pi = mcts.run(state, temp=0.0)   # greedy best move
            action = int(np.argmax(pi))
            move_uci = game.encoder.policy_index_to_move(action)
            print(f"\nMove {move_num}")
            print(f"Model plays {move_uci}")
            board.push_uci(move_uci)
        move_num += 1

    # game over
    print("Final position:")
    print(board.unicode())
    outcome = board.outcome()
    print(f"Game over: {outcome.termination.name}, result = {board.result()}")

    engine.quit()

def play_against_model(game, model, args, human_color=chess.WHITE, num_searches=200):
    """
    Let a human play against the trained AlphaZero model.

    Args:
        game:        instance of ChessGame
        model:       your ResNetAlphaZero instance (already loaded / trained)
        args:        dict of hyper‑parameters (must include C, dirichlet_alpha, dirichlet_epsilon)
        human_color: chess.WHITE or chess.BLACK
        num_searches:number of MCTS sims for each AI move
    """
    # Create a fresh MCTS for interactive play
    play_args = args.copy()
    play_args['num_searches'] = num_searches
    mcts = MCTS(game, model, play_args)

    state = game.get_initial_state()
    print("\nStarting a new game! You are playing",
          "White" if human_color is chess.WHITE else "Black")

    move_num = 1
    while True:
        print(state.board.unicode(), "\n")

        # 1) Human's turn
        if state.board.turn == human_color:
            legal = [m.uci() for m in state.board.legal_moves]
            print(f"Move {move_num}")
            move_uci = None
            while move_uci not in legal:
                print("Valid moves:", legal)
                move_uci = input("Your move (UCI): ").strip()
                if move_uci not in legal:
                    print(" → Illegal move!")
            state.board.push_uci(move_uci)

        # 2) AI's turn
        else:
            print("AI is thinking...", end="", flush=True)
            # run MCTS, temperature=0 for greedy best move
            pi = mcts.run(state, temp=0.0)
            action = int(np.argmax(pi))
            move = game.encoder.policy_index_to_move(action)
            print(f"\nMove {move_num}")
            print(f"\rAI plays {move}     ")
            state = game.get_next_state(state, action, None)

        # Check for game over
        if state.board.is_game_over():
            result = state.board.result()
            outcome = state.board.outcome()
            term = outcome.termination.name if outcome else "unknown"
            print("\nFinal position:\n")
            print(state.board.unicode())
            print(f"\nGame over: {term}, result = {result}")
            break
        move_num += 1

def main():
    # --- then at the bottom of your script, after loading model & optimizer: ---
    # Hyperparameters inspired by the AlphaZero chess paper:
    args = {
        'num_residual_blocks': 20,           # 20 residual blocks (as used in the paper)
        'num_hidden_channels': 256,          # 256 filters per layer
        'num_searches': 800,                # 800 MCTS simulations per move (adjust based on hardware)
        'C': 1.41,                            # Exploration constant (C_puct)
        'dirichlet_epsilon': 0.1,           # Weight of Dirichlet noise in root priors
        'dirichlet_alpha': 0.3,              # Shape parameter for Dirichlet noise
        'play_mode': True
    }

    game = ChessGame()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = ResNetAlphaZero(game, args['num_residual_blocks'], args['num_hidden_channels'], device)
    model.eval()

    load = choose_and_load_checkpoint(model, mode=args['play_mode'], num_residual_blocks=args['num_residual_blocks'], num_hidden_channels=args['num_hidden_channels'], device=device)
    if load == -1:
        return
    choice = input("Play against AI (T) or Test AI against Stockfish (F): ")
    if choice == "T":
        # now hand off to interactive play:
        play_against_model(game, model, args, human_color=chess.WHITE, num_searches=1000)
    else:
        engine_path = "/usr/games/stockfish"
        play_model_vs_stockfish(
        game, model, args,
        stockfish_path=engine_path,
        stockfish_time=0.5,      # half second per Stockfish move
        stockfish_skill_level=20,
        model_searches=1000,
        stockfish_plays_white=True)

if __name__ == "__main__":
    main()

In [ ]:
class SPG:
    """Helper class to hold state for one Self-Play Game."""
    def __init__(self, game):
        self.state = game.get_initial_state() # Current ChessState
        self.memory = [] # Stores history for this game: [(encoded_state, action_probs), ...]
        self.root = None # Root Node of the MCTS tree for the current move
        self.node = None # Current node during MCTS search (used to track expansion target)

class MCTSParallel:
    """Runs MCTS search in parallel across multiple games using batch NN evaluation."""
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model
        # Get device from model
        self.device = next(model.parameters()).device

    @torch.no_grad()
    def search(self, states, spGames): # states is a list of current ChessState objects
        """Performs one batch of MCTS simulations across the provided games."""

        # --- Root Policy/Value Evaluation (Batched) ---
        # 1. Encode all current states individually
        encoded_states_list = [self.game.get_encoded_state(s) for s in states]
        # 2. Stack them into a batch
        encoded_batch = np.stack(encoded_states_list)
        # 3. Convert to tensor
        encoded_batch_tensor = torch.tensor(encoded_batch, dtype=torch.float32, device=self.device)

        # 4. Get policy logits and values from the model
        policy_logits, _ = self.model(encoded_batch_tensor) # We only need policy for root expansion priors
        policy_batch = torch.softmax(policy_logits, dim=1).cpu().numpy()

        # 5. Apply Dirichlet noise
        noise = np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size, size=policy_batch.shape[0])
        policy_batch = (1 - self.args['dirichlet_epsilon']) * policy_batch + self.args['dirichlet_epsilon'] * noise

        # 6. Initialize root nodes and expand them
        for i, spg in enumerate(spGames):
            root_policy = policy_batch[i]
            current_state = states[i] # The ChessState object for this game

            valid_moves = self.game.get_valid_moves(current_state)
            root_policy *= valid_moves # Mask invalid moves
            policy_sum = np.sum(root_policy)
            if policy_sum > 1e-8:
                root_policy /= policy_sum # Normalize
            else: # Handle edge case (e.g., no valid moves?)
                num_valid = np.sum(valid_moves)
                if num_valid > 0: root_policy = valid_moves / num_valid
                else: root_policy = np.zeros(self.game.action_size, dtype=np.float32) # Should not happen if game not ended

            # Create root node (visit_count=0, W=0 initially)
            spg.root = Node(self.game, self.args, current_state, parent=None, action_taken=None, prior=0)
            # Expand the root using the calculated policy
            spg.root.expand(root_policy)

        # --- MCTS Simulation Loop ---
        for _ in range(self.args['num_mcts_sims']): # Number of simulations per move decision
            # -- Selection Phase (Done individually per game tree) --
            for spg in spGames:
                spg.node = None # Reset node marker for expansion
                node = spg.root

                # Traverse down the tree using UCB (select method)
                while node.is_fully_expanded():
                    if not node.children: break # Safety break if node has no children unexpectedly
                    node = node.select()
                    if node is None: # Error in selection?
                         print(f"Warning: Node selection returned None. State: {spg.root.state.board.fen()}")
                         node = spg.root # Go back to root for this sim?
                         break

                # -- Check Terminal State at 'node' --
                # Value is from perspective of player whose turn it IS at node.state
                value, is_terminal = self.game.get_value_and_terminated(node.state)

                if is_terminal:
                    # Backpropagate the terminal value
                    # Pass value relative to the node being evaluated
                    node.backpropagate(value)
                else:
                    # If not terminal and node hasn't been expanded yet (no children),
                    # mark it for NN evaluation in the batch.
                    if not node.children:
                        spg.node = node # Mark for expansion

            # -- Expansion/Evaluation Phase (Batched across games needing it) --
            expandable_indices = [idx for idx, spg in enumerate(spGames) if spg.node is not None]

            if len(expandable_indices) > 0:
                # Get states to evaluate
                states_to_evaluate = [spGames[idx].node.state for idx in expandable_indices]
                # Encode and stack
                encoded_batch_list = [self.game.get_encoded_state(s) for s in states_to_evaluate]
                encoded_batch = np.stack(encoded_batch_list)
                encoded_batch_tensor = torch.tensor(encoded_batch, dtype=torch.float32, device=self.device)

                # Run NN evaluation
                policy_logits, value_batch_tensor = self.model(encoded_batch_tensor)
                policy_batch = torch.softmax(policy_logits, dim=1).cpu().numpy()
                value_batch = value_batch_tensor.cpu().numpy().flatten() # Ensure (N,) shape

                # Process results: Expand nodes and backpropagate NN value
                for i, game_idx in enumerate(expandable_indices):
                    node_to_expand = spGames[game_idx].node
                    nn_policy = policy_batch[i]
                    nn_value = value_batch[i] # Value relative to player turn at node_to_expand.state

                    valid_moves = self.game.get_valid_moves(node_to_expand.state)
                    nn_policy *= valid_moves # Mask
                    policy_sum = np.sum(nn_policy)
                    if policy_sum > 1e-8:
                        nn_policy /= policy_sum # Normalize
                    else: # Handle edge case
                         num_valid = np.sum(valid_moves)
                         if num_valid > 0: nn_policy = valid_moves / num_valid
                         else: nn_policy = np.zeros(self.game.action_size, dtype=np.float32)

                    # Expand the node using the NN policy
                    node_to_expand.expand(nn_policy)

                    # Backpropagate the value estimated by the NN
                    # Pass value relative to the node being evaluated
                    node_to_expand.backpropagate(nn_value)

class AlphaZeroAgentParallel:
    """Agent implementing parallel self-play and training."""
    def __init__(self, game, model, optimizer, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTSParallel(game, args, model)
        # Use a deque for memory efficiency and rolling window
        self.memory = collections.deque(maxlen=args.get('replay_buffer_size', 100000)) # Get buffer size from args

    def selfPlay(self):
        """Runs one batch of parallel self-play games until completion."""
        return_memory = [] # Experience collected from finished games in this batch
        # Initialize Self-Play Game objects
        spGames = [SPG(self.game) for _ in range(self.args['num_parallel_games'])]

        active_game_indices = list(range(self.args['num_parallel_games']))

        while len(active_game_indices) > 0:
            # Get current states from active games only
            current_states = [spGames[i].state for i in active_game_indices]
            active_spGames = [spGames[i] for i in active_game_indices] # Pass only active SPG objects

            # Perform MCTS search step for all active games
            self.mcts.search(current_states, active_spGames)

            # Process results and advance games (iterate based on active_game_indices)
            new_active_indices = []
            finished_indices_this_step = []

            for list_idx, game_idx in enumerate(active_game_indices):
                spg = spGames[game_idx] # Get the correct SPG object
                if spg.root is None: # Safety check if MCTS failed init
                     print(f"Error: Game {game_idx} has no root node after search.")
                     finished_indices_this_step.append(game_idx)
                     continue

                # --- Calculate action probabilities from visit counts ---
                action_probs = np.zeros(self.game.action_size, dtype=np.float32)
                # Sum visit counts of children edges (N(s,a)) as proxy for N(s) at root
                # N(s) = sum(N(s,a) for a in children) = sum(child.visit_count for child in children)
                root_total_visits = sum(child.visit_count for child in spg.root.children)

                if root_total_visits > 0:
                    for child in spg.root.children:
                        if child.action_taken is not None:
                            action_probs[child.action_taken] = child.visit_count / root_total_visits
                else: # No visits recorded (maybe num_searches=0 or error?) -> Use root priors?
                    # Fallback: use uniform valid? (Should ideally not happen with num_searches > 0)
                    valid_m = self.game.get_valid_moves(spg.state)
                    num_v = np.sum(valid_m)
                    if num_v > 0: action_probs = valid_m / num_v
                    else: # Game should have ended if no valid moves
                        print(f"Error: No child visits AND no valid moves for game {game_idx}. State: {spg.state.board.fen()}")
                        finished_indices_this_step.append(game_idx)
                        continue

                # Store experience (encode state here)
                encoded_state = self.game.get_encoded_state(spg.state)
                spg.memory.append((encoded_state, action_probs))

                # --- Determine action using temperature ---
                temp = self.args['temperature'] # Use temperature from args
                # Ensure probs sum to 1 (handle potential float errors)
                prob_sum = np.sum(action_probs)
                if prob_sum < 1e-6: # Check if probs are essentially zero
                    print(f"Warning: Zero probability sum for game {game_idx}, cannot choose action.")
                    # Possibly end this game prematurely?
                    finished_indices_this_step.append(game_idx)
                    continue
                if abs(prob_sum - 1.0) > 1e-5: action_probs /= prob_sum # Renormalize

                if temp < 1e-3: # Use argmax
                    action_idx = np.argmax(action_probs)
                else: # Sample using temperature (if needed, assume action_probs are post-temp for now)
                    # If action_probs are derived from N^(1/T), just sample:
                    action_idx = np.random.choice(self.game.action_size, p=action_probs)
                    # If they are N/sum(N), apply temperature:
                    # temp_probs = action_probs ** (1.0 / temp)
                    # temp_probs /= np.sum(temp_probs)
                    # action_idx = np.random.choice(self.game.action_size, p=temp_probs)

                # --- Update game state ---
                spg.state = self.game.get_next_state(spg.state, action_idx, player=None)

                # --- Check termination ---
                value, is_terminal = self.game.get_value_and_terminated(spg.state)

                if is_terminal:
                    # Game finished, process memory
                    player_turn = 1 # Perspective of first player in history
                    # Value is from perspective of player whose turn it IS now
                    # Convert to outcome relative to White (+1 white win, -1 black win, 0 draw)
                    outcome_white_perspective = value if spg.state.board.turn == chess.WHITE else -value

                    for hist_encoded_state, hist_action_probs in spg.memory:
                        perspective_outcome = outcome_white_perspective * player_turn
                        return_memory.append((hist_encoded_state, hist_action_probs, perspective_outcome))
                        player_turn *= -1 # Flip perspective for next state
                    finished_indices_this_step.append(game_idx)
                else:
                    # Game continues, add index back to active list for next iteration
                    new_active_indices.append(game_idx)

            # Update the list of active game indices
            active_game_indices = new_active_indices

        # Return all collected experiences from finished games in this batch
        return return_memory


    def train(self, memory_batch): # Accepts a batch sampled from memory
        """Trains the model on a single batch of memory."""
        state, policy_targets, value_targets = zip(*memory_batch)

        # Convert to numpy arrays first
        state = np.array(state, dtype=np.float32)
        policy_targets = np.array(policy_targets, dtype=np.float32)
        value_targets = np.array(value_targets, dtype=np.float32).reshape(-1, 1)

        # Convert to tensors
        state_tensor = torch.tensor(state, dtype=torch.float32, device=self.model.device)
        policy_targets_tensor = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
        value_targets_tensor = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)

        # Forward pass - Expect model to output LOGITS for policy
        out_policy_logits, out_value = self.model(state_tensor)

        # Calculate loss
        # Policy loss: Cross-entropy between logits and target probabilities
        # PyTorch's CrossEntropyLoss expects raw logits and class indices, not target probabilities.
        # We need to calculate it manually using log_softmax.
        policy_loss = -torch.sum(policy_targets_tensor * F.log_softmax(out_policy_logits, dim=1), dim=1).mean()
        # Value loss: Mean Squared Error
        value_loss = F.mse_loss(out_value, value_targets_tensor)
        # Total loss
        loss = policy_loss + value_loss # Consider weighting value_loss, e.g., * 0.5 or * 1.0

        # Backward pass and optimization step
        self.optimizer.zero_grad()
        loss.backward()
        # Optional: Gradient clipping
        # torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        self.optimizer.step()

        return loss.item(), policy_loss.item(), value_loss.item() # Return individual losses too


    def learn(self, start_iteration=0):
        """Main training loop using parallel self-play and deque memory."""

        for iteration in range(start_iteration, self.args['num_iterations']):
            iteration_start_time = time.time()
            print(f"\n===== Iteration {iteration + 1}/{self.args['num_iterations']} =====")

            # --- Self-Play Phase ---
            self.model.eval() # Set model to evaluation mode
            print(f"--- Starting Self-Play ({self.args['num_parallel_games']} games) ---")
            # Calculate number of self-play batches needed based on desired total steps
            # This logic might need refinement depending on how num_selfPlay_iterations is defined
            # Let's assume we run parallel games until completion N times
            num_selfplay_runs = self.args.get('num_selfplay_runs_per_iteration', 1) # How many full parallel batches to run

            iteration_memory = []
            for run in range(num_selfplay_runs):
                print(f"  Self-play run {run+1}/{num_selfplay_runs}...")
                run_memory = self.selfPlay() # Runs num_parallel_games games
                iteration_memory.extend(run_memory)

            self.memory.extend(iteration_memory) # Add to the main deque buffer
            iteration_end_time = time.time()
            print(f"Self-Play finished. Generated {len(iteration_memory)} samples in {iteration_end_time - iteration_start_time:.2f}s. Buffer size: {len(self.memory)}")

            # --- Training Phase ---
            if len(self.memory) < self.args['batch_size']:
                 print("Not enough memory to train yet. Skipping training phase.")
                 continue # Skip training if buffer too small

            self.model.train() # Set model to training mode
            print("--- Starting Training ---")
            # Number of training steps this iteration
            num_training_steps = self.args.get('training_steps_per_iteration', len(self.memory) // self.args['batch_size']) # Train on roughly one pass over new data

            total_iter_loss, total_iter_pol_loss, total_iter_val_loss = 0, 0, 0
            actual_steps = 0

            for _ in range(num_training_steps, desc=f"Iteration {iteration+1} Training"):
                if len(self.memory) < self.args['batch_size']: break # Stop if buffer becomes too small
                batch = random.sample(self.memory, self.args['batch_size']) # Sample from deque
                loss, pol_loss, val_loss = self.train(batch) # Train on the batch
                total_iter_loss += loss
                total_iter_pol_loss += pol_loss
                total_iter_val_loss += val_loss
                actual_steps += 1

            if actual_steps > 0:
                print(f"Training finished. Avg Loss: {total_iter_loss / actual_steps:.4f} (Policy: {total_iter_pol_loss / actual_steps:.4f}, Value: {total_iter_val_loss / actual_steps:.4f})")

            # Save model checkpoint
            if (iteration + 1) % self.args.get('save_frequency', 10) == 0: # Use save_frequency from args
                print(f"Saving model and optimizer state for iteration {iteration + 1}...")
                try:
                    checkpoint_path = f"{CHECKPOINT_DIR}/model_{iteration + 1}_{self.args['num_residual_blocks']}_{self.args['num_hidden_channels']}_parallel_chess.pt"
                    save_checkpoint(self.model, self.optimizer, iteration + 1, checkpoint_path)
                except Exception as e:
                    print(f"Error saving checkpoint: {e}")

In [ ]:
import chess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random

class ChessGame:
    """
    Represents the game of chess.
    Handles game logic and state transitions.
    """
    def __init__(self):
        self.row_count = 8
        self.column_count = 8
        self.encoder = ChessEncoder()
        self.action_size = len(self.encoder.index_to_move)

    def get_initial_state(self):
        """Returns the initial state of the game."""
        return ChessState()

    def get_valid_moves(self, state):
        """Returns a binary vector indicating valid moves."""
        valid_moves = np.zeros(self.action_size, dtype=np.uint8)
        legal_moves = [move.uci() for move in state.board.legal_moves]
        for move in legal_moves:
            idx = self.encoder.move_to_policy_index(move)
            if idx is not None:
                valid_moves[idx] = 1
        return valid_moves

    def get_next_state(self, state, action_index, player):
        """Applies an action to the current state and returns the next state."""
        next_state = state.copy()
        move = self.encoder.policy_index_to_move(action_index)
        chess_move = chess.Move.from_uci(move)
        if chess_move in next_state.board.legal_moves:
            next_state.board.push(chess_move)
        return next_state

    def get_reward(self, state):
        """Returns the reward for the current state (1 for win, -1 for loss, 0 for draw)."""
        if state.board.is_checkmate():
            return 1 if state.board.turn else -1
        if state.board.is_stalemate() or state.board.is_insufficient_material() or state.board.is_fifty_moves() or state.board.is_repetition():
            return 0
        return None # Game not terminated

    def get_encoded_state(self, state):
        """Encodes the current game state into a format suitable for the neural network."""
        # This is a placeholder - you'll need a more sophisticated encoding
        board_array = np.zeros((8, 8, 14), dtype=np.float32) # Example: 14 channels (6 for white pieces, 6 for black, 1 for turn, 1 for castling rights)
        piece_map = {'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,
                     'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11}
        for square, piece in state.board.piece_map().items():
            rank = 7 - (square // 8)
            column = square % 8
            piece_type = piece.symbol()
            if piece_type in piece_map:
                layer = piece_map[piece_type]
                board_array[rank, column, layer] = 1
        # Add turn information (channel 12)
        board_array[:, :, 12] = 1 if state.board.turn else 0
        # Add castling rights (channel 13 - simplified)
        board_array[:, :, 13] = 1 if state.board.has_kingside_castling_rights(chess.WHITE) or state.board.has_queenside_castling_rights(chess.WHITE) or state.board.has_kingside_castling_rights(chess.BLACK) or state.board.has_queenside_castling_rights(chess.BLACK) else 0
        return board_array.transpose(2, 0, 1) # Channels first

class ChessState:
    """
    Represents the state of a chess game (using the chess library).
    """
    def __init__(self, board=None):
        self.board = board if board else chess.Board()

    def copy(self):
        return ChessState(self.board.copy())

class ChessEncoder:
    """
    Encodes and decodes chess moves to/from policy indices.
    """
    def __init__(self):
        self.move_to_index = {}
        self.index_to_move = []
        self._create_move_mappings()

    def _create_move_mappings(self):
        """Creates mappings for all possible moves."""
        files = 'abcdefgh'
        ranks = '12345678'
        pieces = ['q', 'r', 'b', 'n']

        for f1 in files:
            for r1 in ranks:
                for f2 in files:
                    for r2 in ranks:
                        move = f"{f1}{r1}{f2}{r2}"
                        self._add_move(move)
                        if r1 == '7' and r2 == '8':
                            for p in pieces:
                                self._add_move(f"{f1}{r1}{f2}{r2}{p}")
                        if r1 == '2' and r2 == '1':
                            for p in pieces:
                                self._add_move(f"{f1}{r1}{r2}{f2}{p}") # Corrected order

    def _add_move(self, move):
        if move not in self.move_to_index:
            self.move_to_index[move] = len(self.index_to_move)
            self.index_to_move.append(move)

    def move_to_policy_index(self, move_uci):
        return self.move_to_index.get(move_uci)

    def policy_index_to_move(self, index):
        return self.index_to_move[index] if 0 <= index < len(self.index_to_move) else None

class NeuralNetwork(nn.Module):
    """
    Abstract base class for the policy-value network.
    """
    def __init__(self):
        super().__init__()

    def forward(self, state):
        raise NotImplementedError("Subclasses must implement the forward pass.")

class ResNetAlphaZero(NeuralNetwork):
    """
    Example ResNet architecture for the AlphaZero policy-value network.
    You'll need to define the specific layers and number of residual blocks.
    """
    def __init__(self, game, num_residual_blocks, num_hidden_channels, device):
        super().__init__()
        self.device = device
        self.conv1 = nn.Conv2d(14, num_hidden_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden_channels)
        self.res_blocks = nn.ModuleList([ResidualBlock(num_hidden_channels) for _ in range(num_residual_blocks)])
        self.policy_head = nn.Sequential(
            nn.Conv2d(num_hidden_channels, 2, kernel_size=1),
            nn.BatchNorm2d(2),
            nn.Flatten(),
            nn.Linear(2 * 8 * 8, game.action_size)
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(num_hidden_channels, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Flatten(),
            nn.Linear(1 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Tanh()
        )
        self.to(device)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        for block in self.res_blocks:
            x = block(x)
        policy = self.policy_head(x)
        value = self.value_head(x)
        return F.softmax(policy, dim=1), value

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        return F.relu(x)

class MCTS:
    """
    Implements the Monte Carlo Tree Search algorithm (iterative version).
    """
    def __init__(self, game, model, args):
        self.game = game
        self.model = model
        self.args = args
        self.Qsa = {}  # Q-values for (state, action)
        self.Nsa = {}  # Visit counts for (state, action)
        self.Ns = {}   # Visit counts for states
        self.Ps = {}   # Prior probabilities for states (from policy network)

    def run(self, initial_state, temp=1):
        """Runs MCTS for a given number of simulations."""
        for _ in range(self.args['num_mcts_sims']):
            self._search(initial_state)

        # Get action probabilities at the root node
        state_str = initial_state.board.fen()
        counts = [self.Nsa.get((state_str, action), 0) for action in range(self.game.action_size)]

        if temp <= 1e-3:
            # Exploitation: Select action with the highest visit count
            best_action = np.argmax(counts)
            probs = [0] * len(counts)
            probs[best_action] = 1
        else:
            # Exploration: Use temperature-based probability with normalized counts
            max_count = max(counts) if counts else 1 # Avoid division by zero
            normalized_counts = [count / max_count for count in counts]
            probs = [count ** (1 / temp) for count in normalized_counts]
            probs_sum = float(sum(probs))
            if probs_sum > 0:
                probs = [x / probs_sum for x in probs]
            else:
                # Fallback to uniform distribution if no visits
                valid_moves = self.game.get_valid_moves(initial_state)
                valid_indices = np.where(valid_moves == 1)[0]
                if valid_indices.size > 0:
                    prob = 1 / valid_indices.size
                    probs = np.zeros(self.game.action_size)
                    for index in valid_indices:
                        probs[index] = prob
                else:
                    probs = [1 / self.game.action_size] * self.game.action_size

        return probs

    def _search(self, initial_state):
        """Performs one MCTS simulation iteratively."""
        state = initial_state
        path = []

        # Selection
        while state.board.fen() in self.Ps:  # While not a leaf node
            state_str = state.board.fen()
            best_action = self.select_action(state)
            path.append((state_str, best_action))
            state = self.game.get_next_state(state, best_action, 1 if state.board.turn else -1)

        # Expansion and Evaluation
        state_str = state.board.fen()
        reward = self.game.get_reward(state)

        if reward is not None:
            value = reward
        else:
            encoded_state = self.game.get_encoded_state(state)
            policy, value = self.model(torch.tensor(encoded_state).unsqueeze(0).float())
            self.Ps[state_str] = policy.detach().cpu().numpy()[0]
            self.Ns[state_str] = 0
            value = -value.item()

        # Backpropagation
        for state_fen, action_index in reversed(path):
            if (state_fen, action_index) not in self.Qsa:
                self.Qsa[(state_fen, action_index)] = 0
                self.Nsa[(state_fen, action_index)] = 0
            self.Qsa[(state_fen, action_index)] = (self.Nsa[(state_fen, action_index)] * self.Qsa[(state_fen, action_index)] + value) / (self.Nsa[(state_fen, action_index)] + 1)
            self.Nsa[(state_fen, action_index)] += 1
            if state_fen in self.Ns:
                self.Ns[state_fen] += 1
            else:
                self.Ns[state_fen] = 1
            value = -value

        return -value

    def select_action(self, state):
        """Selects an action based on the UCB formula."""
        state_str = state.board.fen()
        best_action = -1
        max_ucb = -float('inf')
        valid_moves = self.game.get_valid_moves(state)

        for action_index in range(self.game.action_size):
            if valid_moves[action_index]:
                q_value = self.Qsa.get((state_str, action_index), 0)
                n_sa = self.Nsa.get((state_str, action_index), 0)
                n_s = self.Ns.get(state_str, 1) # Add small constant to avoid division by zero
                prior_p = self.Ps[state_str][action_index]

                ucb = q_value + self.args['C'] * prior_p * math.sqrt(n_s) / (1 + n_sa)

                if ucb > max_ucb:
                    max_ucb = ucb
                    best_action = action_index
        return best_action

class AlphaZeroAgent:
    """
    Orchestrates the self-play and training of the AlphaZero model.
    """
    def __init__(self, game, model, optimizer, args):
        self.game = game
        self.model = model
        self.optimizer = optimizer
        self.args = args
        self.mcts = MCTS(game, model, args)
        self.memory = []

    def self_play(self):
        """Generates one episode of self-play."""
        memory = []
        state = self.game.get_initial_state()
        player = 1
        game_history = []
        move_num = 1

        while True:
            print(f"\nMove {move_num}, Player: {player}")
            print(state.board.unicode())
            temp = 1 if len(game_history) < self.args['temperature_threshold'] else 1e-4 # Explore more in the beginning
            action_probs = self.mcts.run(state, temp=temp)
            memory.append((self.game.get_encoded_state(state), action_probs))

            action_index = np.random.choice(self.game.action_size, p=action_probs)
            move = self.game.encoder.policy_index_to_move(action_index)
            game_history.append(move)

            print(f"Chosen move index: {action_index}, Move: {move}")
            state = self.game.get_next_state(state, action_index, player)
            reward = self.game.get_reward(state)

            if reward is not None:
                print(f"Game terminated. Winner: {player}, Value: {reward}")
                print("Final board:")
                print(state.board.unicode())
                # Assign rewards to the game history
                for hist_state, hist_probs in memory:
                    self.memory.append((hist_state, hist_probs, reward * player)) # Reward from current player's perspective
                break
            player *= -1
            move_num += 1
            self.mcts = MCTS(self.game, self.model, self.args) # Reset MCTS for the next move

    def train_step(self, batch):
        """Performs one training step on a batch of data."""
        states, policy_targets, value_targets = zip(*batch)
        states = torch.tensor(np.array(states)).float()
        policy_targets = torch.tensor(np.array(policy_targets)).float()
        value_targets = torch.tensor(np.array(value_targets)).float().unsqueeze(1)

        self.optimizer.zero_grad()
        policy_preds, value_preds = self.model(states)

        policy_loss = -torch.mean(torch.sum(policy_targets * torch.log(policy_preds + 1e-8), dim=1))
        value_loss = F.mse_loss(value_preds, value_targets)
        total_loss = policy_loss + value_loss

        total_loss.backward()
        self.optimizer.step()
        return total_loss.item(), policy_loss.item(), value_loss.item()

    def train(self):
        """Main training loop."""
        for episode in range(self.args['num_episodes']):
            print(f"Starting episode {episode + 1}")
            self.self_play()
            print(f"Episode {episode + 1} finished, {len(self.memory)} training examples generated.")

            if len(self.memory) > self.args['batch_size']:
                batch_indices = np.random.choice(len(self.memory), self.args['batch_size'], replace=False)
                batch = [self.memory[i] for i in batch_indices]
                loss, policy_loss, value_loss = self.train_step(batch)
                print(f"Training loss: {loss:.4f}, Policy loss: {policy_loss:.4f}, Value loss: {value_loss:.4f}")
                self.memory = [] # Clear memory after each training step for simplicity (can be adjusted)

            if (episode + 1) % self.args['save_frequency'] == 0:
                torch.save(self.model.state_dict(), f"alphazero_chess_model_episode_{episode + 1}.pth")
                torch.save(self.optimizer.state_dict(), f"alphazero_chess_optimizer_episode_{episode + 1}.pth")

def main():
    # Hyperparameters (adjust as needed)
    args = {
        'num_residual_blocks': 19, # Default 8
        'num_hidden_channels': 256, # Default 64
        'num_mcts_sims': 800, # Default 100
        'C': 1.5, # Default 1.0
        'num_episodes': 10, # Default 1000
        'batch_size': 256, # Default 64
        'learning_rate': 0.001,
        'temperature_threshold': 15, # Number of moves before temperature drops
        'save_frequency': 50
    }

    game = ChessGame()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ResNetAlphaZero(game, args['num_residual_blocks'], args['num_hidden_channels'], device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args['learning_rate'], weight_decay=1e-4)
    agent = AlphaZeroAgent(game, model, optimizer, args)

    agent.train()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import chess
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
#import torch_xla
#import torch_xla.core.xla_model as xm
#import torch_xla.distributed.xla_multiprocessing as xmp
from tqdm.notebook import trange
torch.manual_seed(0)

# **AlphaZero AI Model Classes**

In [10]:
class Chess:
    def __init__(self):
        self.row_count = 8
        self.column_count = 8
        self.encoder = ChessEncoder()
        self.action_size = len(self.encoder.index_to_move)

    def __repr__(self):
        return "Chess"

    def get_initial_state(self):
        return ChessState()

    def get_valid_moves(self, state):
        valid_moves = np.zeros(self.action_size, dtype=np.uint8)
        legal_moves = [move.uci() for move in state.board.legal_moves]
        for move in legal_moves:
            idx = self.encoder.move_to_policy_index(move)
            if idx is not None:
                valid_moves[idx] = 1
        return valid_moves

    def get_move_list(self, state):
        return [self.encoder.policy_index_to_move(i) for i, valid in enumerate(self.get_valid_moves(state)) if valid]

    def get_next_state(self, state, action, player):
        next_state = state.copy()
        move = chess.Move.from_uci(action)
        if move in next_state.board.legal_moves:
            next_state.board.push(move)
        return next_state

    def check_win(self, state, action):
        if action is None:
            return False
        return state.board.is_checkmate()

    def get_value_and_terminated(self, state):
        if state.board.is_checkmate():
            return 1, True
        if (state.board.is_stalemate() or
            state.board.is_insufficient_material() or
            state.board.is_fifty_moves() or
            state.board.is_repetition()):
            return 0, True
        return 0, False

    def get_opponent(self, player):
        return -player

    def get_opponent_value(self, value):
        return -value

    def get_encoded_state(self, state):
        return self.encoder.encode_state(state)

    def board_to_array(self, board):
        state = np.zeros((8, 8), dtype=np.int8)
        piece_map = {'P': 1, 'N': 2, 'B': 3, 'R': 4, 'Q': 5, 'K': 6,
                     'p': -1, 'n': -2, 'b': -3, 'r': -4, 'q': -5, 'k': -6}
        for square, piece in board.piece_map().items():
            rank = 7 - (square // 8)
            column = square % 8
            state[rank][column] = piece_map[piece.symbol()]
        return state

class ChessState:
    def __init__(self, board=None, history=None):
        self.board = board if board else chess.Board()
        self.history = history if history is not None else []

    def copy(self):
        return ChessState(self.board.copy(), self.history.copy())

    def apply_move(self, move_uci):
        move = chess.Move.from_uci(move_uci)
        if move in self.board.legal_moves:
            self.history.append(self.board.fen())
            self.board.push(move)
            return True
        return False

class ChessEncoder:
    def __init__(self):
        self.move_to_index = {}
        self.index_to_move = []
        self._create_move_mappings()
        self.num_planes = 119  # As suggested by AlphaZero paper

    def _create_move_mappings(self):
        files = 'abcdefgh'
        ranks = '12345678'
        pieces = ['q', 'r', 'b', 'n']

        # Create basic moves and underpromotions for specific ranks.
        for r1 in ranks:
            for f1 in files:
                for r2 in ranks:
                    for f2 in files:
                        move = f"{f1}{r1}{f2}{r2}"
                        self._add_move(move)
                        # Underpromotions for pawn moves from rank 7 to 8 and rank 2 to 1.
                        if r1 == '7' and r2 == '8':
                            for p in pieces:
                                self._add_move(f"{f1}{r1}{f2}{r2}{p}")
                        if r1 == '2' and r2 == '1':
                            for p in pieces:
                                self._add_move(f"{f1}{r1}{f2}{r2}{p}")

    def _add_move(self, move):
        if move not in self.move_to_index:
            self.move_to_index[move] = len(self.index_to_move)
            self.index_to_move.append(move)

    def move_to_policy_index(self, move_uci):
        return self.move_to_index.get(move_uci)

    def policy_index_to_move(self, index):
        return self.index_to_move[index] if index < len(self.index_to_move) else None

    def encode_state(self, state):
        board = state.board
        history = state.history
        planes = np.zeros((self.num_planes, 8, 8), dtype=np.float32)
        piece_map = {'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,
                     'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11}
        # Consider the last 8 moves plus the current board
        fen_history = history[-8:] + [board.fen()]
        for i, fen in enumerate(fen_history):
            temp_board = chess.Board(fen=fen)
            for square, piece in temp_board.piece_map().items():
                rank = 7 - (square // 8)
                column = square % 8
                piece_idx = piece_map.get(piece.symbol())
                if piece_idx is not None:
                    # Use separate planes for moves played by different players
                    plane_index = i * 12 + piece_idx if i % 2 == 0 else i * 12 + piece_idx + 12
                    if plane_index < self.num_planes:
                        planes[plane_index, rank, column] = 1

        # Additional planes for castling rights, halfmove clock, etc.
        planes[96, :, :] = 1 if board.has_kingside_castling_rights(chess.WHITE) else 0
        planes[97, :, :] = 1 if board.has_queenside_castling_rights(chess.WHITE) else 0
        planes[98, :, :] = 1 if board.has_kingside_castling_rights(chess.BLACK) else 0
        planes[99, :, :] = 1 if board.has_queenside_castling_rights(chess.BLACK) else 0
        planes[100, :, :] = board.halfmove_clock / 100.0
        planes[101, :, :] = board.fullmove_number / 500.0
        planes[102, :, :] = 1 if board.turn == chess.WHITE else 0
        planes[103, :, :] = board.ply() / 1000.0

        side_plane = np.full((8,8), 1 if board.turn == chess.WHITE else 0, dtype=np.float32)
        planes[-1] = side_plane  # put at the last plane
        return planes

    def get_policy_index(self, move):
        """
        Computes a policy index for a given chess.Move object.
        Returns an integer index corresponding to the move.
        For queen promotions, we return None because they are part of the base moves.
        """
        if move.promotion is not None:
            promotion_piece = move.promotion
            if promotion_piece == chess.QUEEN:
                return None
            elif promotion_piece == chess.ROOK:
                return 56 + 8
            elif promotion_piece == chess.BISHOP:
                return 56 + 8 + 1
            elif promotion_piece == chess.KNIGHT:
                return 56 + 8 + 2
        else:
            from_square = move.from_square
            to_square = move.to_square
            rank_diff = chess.square_rank(to_square) - chess.square_rank(from_square)
            file_diff = chess.square_file(to_square) - chess.square_file(from_square)

            # North/South moves
            if file_diff == 0 and abs(rank_diff) <= 7:
                return (abs(rank_diff) - 1) if rank_diff > 0 else 4 * 7 + (abs(rank_diff) - 1)
            # East/West moves
            elif rank_diff == 0 and abs(file_diff) <= 7:
                return 1 * 7 + (abs(file_diff) - 1) if file_diff > 0 else 5 * 7 + (abs(file_diff) - 1)
            # Diagonals NE/SW
            elif abs(rank_diff) == abs(file_diff) and rank_diff != 0:
                return 2 * 7 + (abs(rank_diff) - 1) if rank_diff > 0 else 6 * 7 + (abs(rank_diff) - 1)
            # Diagonals NW/SE
            elif rank_diff == -file_diff and rank_diff != 0:
                return 3 * 7 + (abs(rank_diff) - 1) if rank_diff > 0 else 7 * 7 + (abs(rank_diff) - 1)
            # Knight moves
            elif (abs(rank_diff) == 2 and abs(file_diff) == 1) or (abs(rank_diff) == 1 and abs(file_diff) == 2):
                knight_moves = [(1, 2), (1, -2), (-1, 2), (-1, -2), (2, 1), (2, -1), (-2, 1), (-2, -1)]
                for i, (dr, df) in enumerate(knight_moves):
                    if rank_diff == dr and file_diff == df:
                        return 56 + i
        return None

In [11]:
class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        return F.relu(x)

class ChessResNet(nn.Module):
    def __init__(self, game, num_resBlocks, num_hidden, device):
        super().__init__()
        self.device = device

        # Starting convolution block.
        self.startBlock = nn.Sequential(
            nn.Conv2d(game.encoder.num_planes, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )

        # Backbone: a sequence of residual blocks.
        self.backBone = nn.ModuleList([ResBlock(num_hidden) for _ in range(num_resBlocks)])

        # Policy head produces an output of shape (8 x 8 x 72) = 4608 probabilities.
        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 72, kernel_size=1),
            nn.Flatten()
        )

        # Value head outputs a single value.
        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.row_count * game.column_count, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Tanh()
        )

        self.to(device)

    def forward(self, x):
        x = self.startBlock(x)
        for block in self.backBone:
            x = block(x)
        policy_out = self.policyHead(x)
        value = self.valueHead(x)
        return policy_out, value

In [12]:
class Node:
    def __init__(self, game, args, state, parent=None, action_taken=None, prior=0, visit_count=0):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken
        self.prior = prior
        self.children = {}  # Dictionary: action index -> child Node
        self.visit_count = visit_count
        self.value_sum = 0

    def is_fully_expanded(self):
        return len(self.children) == np.count_nonzero(self.game.get_valid_moves(self.state))

    def select(self):
        best_child = None
        best_ucb = -np.inf
        for child in self.children.values():
            ucb = self._get_ucb(child)
            if ucb > best_ucb:
                best_ucb = ucb
                best_child = child
        return best_child

    def _get_ucb(self, child):
        if child.visit_count == 0:
            q_value = 0
        else:
            q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
        return q_value + self.args['C'] * (math.sqrt(self.visit_count) / (child.visit_count + 1)) * child.prior

    def expand(self, policy):
        # Expand for every move that has nonzero probability.
        for action_index, prob in enumerate(policy):
            if prob > 0 and action_index not in self.children:
                move = self.game.encoder.policy_index_to_move(action_index)
                if move and move in [m.uci() for m in self.state.board.legal_moves]:
                    child_state = self.state.copy()
                    if child_state.apply_move(move):
                        self.children[action_index] = Node(self.game, self.args, child_state, self, action_index, prob)

    def backpropagate(self, value):
        self.value_sum += value
        self.visit_count += 1
        if self.parent:
            self.parent.backpropagate(self.game.get_opponent_value(value))

In [13]:
class MCTS:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model

    @torch.no_grad()
    def search(self, state):
        root = Node(self.game, self.args, state, visit_count=1)
        # Encode state for network evaluation.
        encoded_state = self.game.get_encoded_state(state)
        encoded_state = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device).unsqueeze(0)
        policy_logits, _ = self.model(encoded_state)
        # Ensure policy_logits is reshaped to (1, 4608)
        policy = torch.softmax(policy_logits.view(1, -1), dim=1).squeeze(0).cpu().numpy()  # Shape: (4608,)

        valid_moves = self.game.get_valid_moves(state)  # Expected shape: (4608,)
        # Mask illegal moves.
        masked_policy = policy * valid_moves
        if np.sum(masked_policy) > 0:
            masked_policy /= np.sum(masked_policy)
        else:
            indices = np.where(valid_moves == 1)[0]
            masked_policy = np.zeros_like(policy)
            if indices.size > 0:
                masked_policy[indices] = 1.0 / len(indices)

        noise = np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size)
        policy_with_noise = ((1 - self.args['dirichlet_epsilon']) * masked_policy + self.args['dirichlet_epsilon'] * noise)
        policy_with_noise /= np.sum(policy_with_noise)
        root.expand(policy_with_noise)

        for _ in range(self.args['num_searches']):
            node = root
            search_path = [node]
            while node.is_fully_expanded():
                node = node.select()
                if node is None:
                    break  # Node has no children, likely terminal.
                search_path.append(node)

            if node:
                value, is_terminal = self.game.get_value_and_terminated(node.state)
                value = self.game.get_opponent_value(value)
            else:
                value, is_terminal = 0, True

            if not is_terminal:
                # Evaluate node in one go.
                encoded_state = self.game.get_encoded_state(node.state)
                encoded_state = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device).unsqueeze(0)
                policy_logits, value_tensor = self.model(encoded_state)
                policy = torch.softmax(policy_logits.view(1, -1), dim=1).squeeze(0).cpu().numpy()
                valid_moves = self.game.get_valid_moves(node.state)
                masked_policy = policy * valid_moves
                if np.sum(masked_policy) > 0:
                    masked_policy /= np.sum(masked_policy)
                else:
                    indices = np.where(valid_moves == 1)[0]
                    masked_policy = np.zeros_like(policy)
                    if indices.size > 0:
                        masked_policy[indices] = 1.0 / len(indices)
                value = value_tensor.item()
                node.expand(masked_policy)
            self._backpropagate(search_path, value)

        action_probs = np.zeros(self.game.action_size)
        for action_index, child in root.children.items():
            action_probs[action_index] = child.visit_count
        action_probs /= np.sum(action_probs)
        return action_probs

    def _backpropagate(self, search_path, value):
        for node in reversed(search_path):
            node.backpropagate(value)
            value = self.game.get_opponent_value(value)

In [14]:
class AlphaZero:
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTS(game, args, model)
        self.memory = []

    def selfPlay(self):
        memory = []
        player = 1
        state = self.game.get_initial_state()
        move_num = 1

        while True:
            print(f"\nMove {move_num}, Player: {player}")
            print(state.board.unicode())
            action_probs = self.mcts.search(state)
            memory.append((state, action_probs, player))

            # Temperature adjustment and safeguard.
            temperature_action_probs = (action_probs ** (1 / self.args['temperature']) + 1e-8)
            temperature_action_probs /= np.sum(temperature_action_probs)

            action_index = np.random.choice(self.game.action_size, p=temperature_action_probs)
            action = self.game.encoder.policy_index_to_move(action_index)
            print(f"Chosen move index: {action_index}, Move: {action}")
            state = self.game.get_next_state(state, action, player)
            value, is_terminal = self.game.get_value_and_terminated(state)

            if is_terminal:
                print(f"Game terminated. Winner: {player}, Value: {value}")
                print("Final board:")
                print(state.board.unicode())
                return_memory = []
                for hist_state, hist_action_probs, hist_player in memory:
                    outcome = value if hist_player == player else self.game.get_opponent_value(value)
                    return_memory.append((self.game.get_encoded_state(hist_state),
                                          hist_action_probs,
                                          outcome))
                return return_memory

            player = self.game.get_opponent(player)
            move_num += 1

    def train(self, memory):
        random.shuffle(memory)
        epoch_loss = 0
        num_batches = (len(memory) + self.args['batch_size'] - 1) // self.args['batch_size']
        for batchIdx in range(num_batches):
            batch = memory[batchIdx * self.args['batch_size']: (batchIdx + 1) * self.args['batch_size']]
            states, policy_targets, value_targets = zip(*batch)
            states = torch.tensor(np.array(states), dtype=torch.float32, device=self.model.device)
            policy_targets = torch.tensor(np.array(policy_targets), dtype=torch.float32, device=self.model.device)
            value_targets = torch.tensor(np.array(value_targets).reshape(-1, 1), dtype=torch.float32, device=self.model.device)

            out_policy, out_value = self.model(states)
            log_policy = F.log_softmax(out_policy.view(-1, 4608), dim=1)
            policy_loss = -torch.mean(torch.sum(policy_targets * log_policy, dim=1))
            value_loss = F.mse_loss(out_value, value_targets)
            loss = policy_loss + value_loss

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step() if torch.cuda.is_available() or torch.cpu.is_available() else xm.optimizer_step(optimizer)
            epoch_loss += loss.item()
        print(f"Loss: {epoch_loss / (num_batches if num_batches > 0 else 1)}")

    def learn(self):
        for iteration in range(self.args['num_iterations']):
            self.model.eval()
            for selfPlay in trange(self.args['num_selfPlay_iterations']):
                print(f"SelfPlay {selfPlay + 1}")
                self.memory.extend(self.selfPlay())
            self.model.train()
            for epoch in trange(self.args['num_epochs']):
                print(f"Epoch {epoch + 1}", end=" ")
                self.train(self.memory)
            torch.save(self.model.state_dict(), f"model_{iteration + 1}_{self.game}.pt")
            torch.save(self.optimizer.state_dict(), f"optimizer_{iteration + 1}_{self.game}.pt")
            self.memory = []

    @torch.no_grad()
    def play_against_human(self, human_color=chess.WHITE):
        board = chess.Board()
        self.model.eval()
        while not board.is_game_over():
            print("\n" + board.unicode())
            if board.turn == human_color:
                legal_moves = [move.uci() for move in board.legal_moves]
                while True:
                    human_move = input("Your move (UCI format, e.g., e2e4): ")
                    if human_move in legal_moves:
                        board.push_uci(human_move)
                        break
                    else:
                        print("Invalid move. Try again.")
            else:
                state = ChessState(board=board, history=[])
                encoded_state = self.game.get_encoded_state(state)
                encoded_tensor = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device).unsqueeze(0)
                policy_logits, _ = self.model(encoded_tensor)
                policy = torch.softmax(policy_logits.view(1, -1), dim=1).squeeze(0).cpu().numpy()
                valid_moves = self.game.get_valid_moves(state)
                masked_policy = policy * valid_moves
                if np.sum(masked_policy) > 0:
                    masked_policy /= np.sum(masked_policy)
                else:
                    indices = np.where(valid_moves == 1)[0]
                    masked_policy = np.zeros_like(policy)
                    if indices.size > 0:
                        masked_policy[indices] = 1.0 / len(indices)

                best_move_index = np.argmax(masked_policy)
                if best_move_index != -1:
                    best_move = self.game.encoder.policy_index_to_move(best_move_index)
                    if best_move:
                        print(f"AI's move: {best_move}")
                        board.push_uci(best_move)
                    else:
                        print("AI chose an invalid move.")
                        break
                else:
                    print("AI could not find a valid move.")
                    break
        print("\nGame over. Result:", board.result())

In [15]:
class MCTSParallel:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model

    @torch.no_grad()
    def search(self, states, spGames):
        # define root
        encoded_state = self.game.get_encoded_state(np.stack([self.game.board_to_array(state.board) for state in states]))
        encoded_state = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device)

        policy, _ = self.model(encoded_state)
        policy = torch.softmax(policy, axis=1).cpu().numpy()

        noise = np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size, size=policy.shape[0])
        policy = (1 - self.args['dirichlet_epsilon']) * policy + self.args['dirichlet_epsilon'] * noise

        for i, spg in enumerate(spGames):
            spg_policy = policy[i]
            valid_moves = self.game.get_valid_moves(states[i])
            spg_policy *= valid_moves
            spg_policy /= np.sum(spg_policy)

            spg.root = Node(self.game, self.args, states[i], visit_count=1)
            spg.root.expand(spg_policy)

        for _ in range(self.args['num_searches']):
            for spg in spGames:
                spg.node = None
                node = spg.root

                while node.is_fully_expanded():
                    node = node.select() # Selection

                value, is_terminal = self.game.get_value_and_terminated(node.state, node.action_taken)
                value = self.game.get_opponent_value(value)

                if is_terminal:
                    node.backpropagate(value) # Backpropagation
                else:
                    spg.node = node

            expandable_spGames = [mappingIdx for mappingIdx in range(len(spGames)) if spGames[mappingIdx].node is not None]
            if len(expandable_spGames) > 0:
                states = np.stack([self.game.board_to_array(spGames[mappingIdx].node.state.board) for mappingIdx in expandable_spGames])
                encoded_state = self.game.get_encoded_state(states)
                encoded_state = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device)

                policy, value = self.model(encoded_state)
                policy = torch.softmax(policy, axis=1).cpu().numpy()
                value = value.cpu().numpy()

            for i, mappingIdx in enumerate(expandable_spGames):
                node = spGames[mappingIdx].node
                spg_policy, spg_value = policy[i], value[i]

                valid_moves = self.game.get_valid_moves(node.state)
                spg_policy *= valid_moves
                spg_policy /= np.sum(spg_policy)

                node.expand(spg_policy) # Expansion
                node.backpropagate(spg_value) # Backpropagation

In [16]:
class AlphaZeroParallel:
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTSParallel(game, args, model)

    def selfPlay(self):
        return_memory = []
        player = 1
        spGames = [SPG(self.game) for _ in range(self.args['num_parallel_games'])]

        while len(spGames) > 0:
            states = np.stack([spg.state for spg in spGames])
            self.mcts.search(states, spGames)

            for i in range(len(spGames))[::-1]:
                spg = spGames[i]

                action_probs = np.zeros(self.game.action_size) # return visit_counts
                for child in spg.root.children:
                    action_probs[child.action_taken] = child.visit_count
                action_probs /= np.sum(action_probs)

                spg.memory.append((spg.root.state, action_probs, player))

                temperature_action_probs = action_probs ** (1 / self.args['temperature'])
                temperature_action_probs /= np.sum(temperature_action_probs)

                action = np.random.choice(self.game.action_size, p=temperature_action_probs)

                action = self.game.encoder.policy_index_to_move(action)

                spg.state = self.game.get_next_state(spg.state, action, player)

                value, is_terminal = self.game.get_value_and_terminated(spg.state, action)

                if is_terminal:
                    for hist_neutral_state, hist_action_probs, hist_player in spg.memory:
                        hist_outcome = value if hist_player == player else self.game.get_opponent_value(value)
                        return_memory.append((self.game.get_encoded_state(hist_neutral_state),
                                            hist_action_probs,
                                            hist_outcome))
                    del spGames[i]
            player = self.game.get_opponent(player)

        return return_memory

    def train(self, memory):
        random.shuffle(memory)
        epoch_loss = 0
        for batchIdx in range(0, len(memory), self.args['batch_size']):
            sample = memory[batchIdx:min(len(memory) - 1, batchIdx + self.args['batch_size'])]
            state, policy_targets, value_targets = zip(*sample)

            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)

            state = torch.tensor(state, dtype=torch.float32, device=self.model.device)
            policy_targets = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
            value_targets = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)

            out_policy, out_value = self.model(state)

            log_policy = F.log_softmax(out_policy, dim=1)
            policy_loss = -torch.mean(torch.sum(policy_targets * log_policy, dim=1))

            value_loss = F.mse_loss(out_value, value_targets)
            loss = policy_loss + value_loss

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            epoch_loss += loss.item()
        print(f"loss: {epoch_loss/len(sample)}")

    def learn(self):
        for iteration in range(self.args['num_iterations']):
            memory = []

            self.model.eval()
            for selfPlay_iteration in trange(self.args['num_selfPlay_iterations'] // self.args['num_parallel_games']):
                memory.extend(self.selfPlay())

            self.model.train()
            for epoch in trange(self.args['num_epochs']):
                print(f"Epoch {epoch + 1}", end=" ")
                self.train(memory)

            torch.save(self.model.state_dict(), f"model_{iteration + 1}_{self.game}.pt")
            torch.save(self.optimizer.state_dict(), f"optimizer_{iteration + 1}_{self.game}.pt")

class SPG:
    def __init__(self, game):
        self.state = game.get_initial_state()
        self.memory = []
        self.root = None
        self.node = None

# **Train AI Model**
Time Duration:

*   CPU: 62 hrs
*   GPU: 5 hrs


In [17]:
game = Chess()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") \
        if torch.cuda.is_available() or torch.cpu.is_available() else xm.xla_device()
model = ChessResNet(game, 19, 256, device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

args = {'C': 2,
        'num_searches': 200, # Reduced for speed 800
        'num_iterations': 20,
        'num_selfPlay_iterations': 3, # Reduced for testing 1000
        'num_parallel_games': 1000,
        'num_epochs': 2, # Reduced for testing 4
        'batch_size': 64, # Reduced for testing 256
        'temperature': 1.25,
        'dirichlet_epsilon': 0.25,
        'dirichlet_alpha': 0.3
        }

alphazero = AlphaZero(model, optimizer, game, args)
alphazero.learn()

# Example of playing against the trained AI
# alphazero.play_against_human(human_color=chess.BLACK)

  0%|          | 0/3 [00:00<?, ?it/s]

SelfPlay 1

Move 1, Player: 1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ♟ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ♙ ♙ ♙ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Chosen move index: 1150, Move: g2g4

Move 2, Player: -1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ♟ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ♙ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ♙ ♙ ♙ ♙ ♙ ⭘ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Chosen move index: 3853, Move: f7f6

Move 3, Player: 1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ⭘ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ♟ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ♙ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ♙ ♙ ♙ ♙ ♙ ⭘ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Chosen move index: 665, Move: b2b4

Move 4, Player: -1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ⭘ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ♟ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ♙ ⭘ ⭘ ⭘ ⭘ ♙ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ⭘ ♙ ♙ ♙ ♙ ⭘ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Chosen move index: 3748, Move: e7e5

Move 5, Player: 1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ⭘ ⭘ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ♟ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ♟ ⭘ ⭘ ⭘
⭘ ♙ ⭘ ⭘ ⭘ ⭘ ♙ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ⭘ ♙ ♙ ♙ ♙ ⭘ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Chosen move index: 859, Move: d2d4

Move 6, Player: -1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ⭘ ⭘ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ♟ ⭘ ⭘
⭘ ⭘ ⭘ 

 33%|███▎      | 1/3 [10:17<20:34, 617.13s/it]

Chosen move index: 2061, Move: e4f2
Game terminated. Winner: -1, Value: 1
Final board:
⭘ ⭘ ♗ ⭘ ⭘ ♝ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♝ ♟ ⭘ ♘ ⭘ ♙ ⭘ ♜
⭘ ⭘ ⭘ ♚ ⭘ ⭘ ♟ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ♜ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ♞ ⭘ ♙
⭘ ⭘ ⭘ ⭘ ♖ ⭘ ⭘ ♔
SelfPlay 2

Move 1, Player: 1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ♟ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ♙ ♙ ♙ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Chosen move index: 568, Move: a2a4

Move 2, Player: -1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ♟ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
♙ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ♙ ♙ ♙ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Chosen move index: 3845, Move: f7f5

Move 3, Player: 1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ⭘ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ♟ ⭘ ⭘
♙ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ♙ ♙ ♙ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Chosen move index: 665, Move: b2b4

Move 4, Player: -1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ♟ ♟ ♟ ⭘ ♟ ♟
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ♟ ⭘ ⭘
♙ ♙ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘ ⭘
⭘ ⭘ ♙ ♙ ♙ ♙ ♙ ♙
♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
Chosen move index: 3562, Move: c7c6

Move 5, Player: 1
♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
♟ ♟ ⭘ ♟

 33%|███▎      | 1/3 [12:54<25:48, 774.11s/it]


KeyboardInterrupt: 